# Deney 06 — Render Pipeline (Notebook 01+02+03 çıktılarını birleştirir)

**Amaç:** Onaylanmış an + script + seslendirmeyi kullanarak dikey (9:16) bir YouTube Shorts videosu üretmek: kaynak aralığı kes, dikey kadraj uygula, seslendirmeyi orijinal sesle miksle (ducking), altyazı yak, kaliteli kodlama ile dışa aktar.

**Çalıştırma sırası:** Notebook 01 → an onayı → Notebook 02 → script onayı → Notebook 03 → seslendirme onayı → **bu notebook**. Alternatif olarak `00_youtube_automation_master.ipynb` tek başına tüm fazları çalıştırabilir.

**Bu notebook hiçbir yere yükleme/yayınlama yapmaz.** Çıktı sadece `results/render/final_short.mp4` olarak yerelde/Colab oturumunda kalır.

**Tek kaynak politikası (düzeltildi):** Bu notebook artık ffmpeg filtergraph'larını, altyazı üretimini, doğrulamayı ve onay kapısı mantığını **kendi içinde tekrar tanımlamaz**. Hepsi `local_pipeline/render_utils.py` içindeki kanonik fonksiyonlardan gelir — master notebook (`00_...`) ve `verify_local.py` de **aynı** fonksiyonları çağırır, yani üç ayrı kopya yoktur. Modül aşağıdaki bootstrap hücresiyle yüklenir (repo → Drive → `PROJECT_REPO_URL` → gömülü hash-doğrulamalı yedek).

**Onay politikası (üretimden sonra, içeriğe bağlı):** `run_render_pipeline()` **onay argümanı almaz** ve asla `approved` üretemez — yeni render her zaman `review_required` (teknik doğrulama başarısızsa `failed`) olur. Onay, videoyu **izledikten sonra** bölüm 5'teki ayrı hücrede, dosyanın SHA-256'sı doğrulanarak verilir. Yeniden render alırsan hash değişir ve önceki onay otomatik geçersizleşir. `render_validation` bir **teknik** kontrol sonucudur, insan onayı değildir.

In [ ]:
!pip -q install imageio-ffmpeg faster-whisper pandas > /dev/null
print('Kurulum tamam.')

In [ ]:
import os, sys, json
from pathlib import Path

if 'google.colab' not in sys.modules:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        venv_site_packages = candidate / '.venv' / 'Lib' / 'site-packages'
        if venv_site_packages.is_dir():
            import site
            site.addsitedir(str(venv_site_packages))
            break

IS_COLAB = 'google.colab' in sys.modules or os.path.isdir('/content')
PROJECT_ROOT = Path.cwd()
RESULTS_DIR = Path('/content/results') if IS_COLAB else PROJECT_ROOT / 'results'
RENDER_DIR = RESULTS_DIR / 'render'
RENDER_DIR.mkdir(parents=True, exist_ok=True)

STATUS_VALUES = {
    'experimental', 'development_placeholder', 'review_required',
    'approved', 'insufficient_data', 'failed', 'production_ready',
}
print('Sonuc dizini:', RESULTS_DIR)


## `local_pipeline` kanonik modüllerini yükle (tek kaynak)

In [ ]:
# @title Yerel pipeline yukleyici (otomatik - acmaniz gerekmez)
import sys, os, tempfile, subprocess, base64, hashlib, zlib
from pathlib import Path

PROJECT_REPO_URL = ''  # opsiyonel: gercek bir git repo URL'si verirsen modul(ler) oradan klonlanir

_LOCAL_PIPELINE_MODULE_NAMES = ['render_utils', 'video_analysis', 'approval']

# Her modul icin: (base64 kaynak, SHA-256 hash). Hash, gomulu payload'in
# bozulmadigini/kesilmedigini dogrulamak icin kullanilir -- eslesmezse acik
# bir hata verilir, asla sessizce bozuk kod calistirilmaz.
_LOCAL_PIPELINE_PAYLOADS = {
    'render_utils': {
        'sha256': 'b35f4b1004b927d6b69d715ed563a41f6c14381fd82ed2fdd02e10630b3e04c9',
        'encoding': 'zlib+b85',
        'data': (
            'c-rl~+jiSXmLU4BuZU8fX^@#9P05#Z(Z#H#wB<@qIV~?snVxE;h5|{DL<t1gxJafoTx<SdUgmS=Yvu>eFPwde2t)(`O1@OrsqV_|'
            '%A|mZjkxUCvG03-fB#)r=D{Qm#mkqgb+`~&n9Rbgw~p3f93`PxhVeSga*;2CES!nSMhu>exzSiP>PPEMG7+7dWi(yxMF~F3MUaT_'
            '-^%DZh{L4t#q%@{CcOwcuCaGgh-tdsh;%NBWt59JT}0C^bef6lFpK7!ahy(rxWC@)F<cR?)@fGc+<cYJN*MGk%ED<8Z^S&!;6*5?'
            'lbnReSLszKiZCy_{k^?6DfCPhFrzFK<vLD-SuUX4x}3yOzN8Uvs7YVEDnyz9*et|G40ImF)Mh4bvIv0gnU72QU?AX049l07>vbH#'
            'dZlHtE(@^^ie-Oqe}8{(o~0`>9?#37%);?l&I-ULX%Q4rn&f+X^4T)TmvJ;vfBrR36ZJa_)vtV67E!GKDktkKorZa?o)k-*15Ap)'
            'JPV4jh*qKcuKL>-`0tNt5^_5nFnn7zdkcT=?VX+-o&9n;{`KhfFUO~1ByRTv{Ief^01ScNNfE^RzSy6I*I}Hl@xgc<2h(tw#;`j0'
            'Dhscp@CNV)5ChO@-)9KH8b*DMe~bMn$;<gXnnu_ih8N(&c>t?Suc6_roZ>=_abg<)v4xTS^5fa7v)9Mt_dmTleS7?F{PW3+<JUAs'
            'uZWV3=L;`i!QSAHRXB^v75<8YY!UXZANF7~CaDL2o*%tEdv)?=eD>4v&&T7p?@nI6dTq?rn}zeBj0<llAJ9V>6Jo&_5I2W`!OH``'
            'dOJ^HB1y0c;koxB{A=)QDNchV_uMvl^by;9e`vf2V5-;Qc$&s(HV*O}Vf@Qa1Ni49{9}Tc1bIWdmoMFRtKh@<Ce3Ep<dO4y3Wpq@'
            'KO9&uvPG1Pui^ROq4@%EJj|x>S$Oj@i1W~R7l%aw6e3>-Q@}lFAUkgPKi&e?W1P;%fS!PJ`CaRI*tVWO2iY`?<1!bg&s!IC@ONL('
            '!>6CRpl4aRdp-Lm$RY!Tnr6|O4hF2>YK=JB(JiReiy&O3i8zDr2iYb}eDQ7>gvllg%xAwwd6;c-^U=?nJjkK|PK*ukr?kvNv*CN7'
            'wPBX#zIc%Z3m|3%?5;2V3}gPy%f)M;p765<=T(NoB2<9y2k{0zh9e2#fNlT*bnC9W#fT?0TsVE+dlxQB=-e-gc^xg5!Qta4*kkZ;'
            '@aR!6dGh%1@abgm`0=CZ?}NwFgYe12+5GVLvq$0Mr@{9JPal6D9v&P%9UKI~{QD<QXY<*?;K^h*AJjmXS&VR(MX}C@4<2N}O@9#;'
            '%W?wilue-p5Vig^T|HQ&=^_pvV5j`SDu|K?={$xXat8VI!98XZN7FC?BsCY|<mKyr@u6sH_7w-$@RbQ%fk2~yD#rmGF}y<XdvJI{'
            'fTN&EAieudQ9)I6_yp+o7$xEPM*KNk!*$E-$p^Tkrx9`@t2H1vT(A$aG>#|1^h%_3mipqTEYcNxb1MFHawamk(2<Eq(i;g67B45('
            'W0eA50T*x)6>-?Z%Zd0ZlR^ne*+ejb>#pBO3deB3KZfyV-!RX)dkZam^7QXd3%~!`wD6#RxQj0Eiu8U(5(YbR2Thninmv6ud-!N}'
            'Fn==r{?Wm~?;nPbg6aIx1c<`?_m3YQgpY$K-%p+d0nv@wU=Vyin|yy5JPHT%hmXECO<;6*z?1kdkAJ*bEQ|m0_{HCs7RW(;HW{!c'
            ')w(mEcK92|v?ccNPe;JaoxB9D?%m0o<5S7q{rUJW?@!*nK<=(n9YvN%m@9#cYDrfdJp*xrE;W2ys5=k;xRJM>{H7P-b;dC8^{l!I'
            '=})?a@mD5qUxE?AnrHJ6S=7Dm-rgQ?n!x7gX^bZ{;ENr`unIfKJ>wT={P$4gMFw;50-#g?igxcqF~)!CcV1WYegJW4S`3-xM#+zW'
            'gU~Vy;nS-qiE<=!@?)A-kvyPDC4CabCWRy@)F4DXD)=0H)Ey;u;SqNG@H$#|y8Sr43A0X@x<_*Z!tc~<h8h{H_aG~YC=t!f`q1Jn'
            'vH;i_<SEMVILp$^W{L2h`Tma)NUW<6c2iGYuI#P;=I^rYGqK;$c@<tyQNp9u0@)8b<sdSJvrhByx?3AcSN|n<d*-9q33m}Jr2af$'
            'h}r^PPv(h2;JT~%CTRvT4iNg;SR#Sk3lk-1BRqTY@gKP3rc2tr1X|h4)VT^bNY+WrR=;5b?l@PlXGY`t=nQTS|9OM#-(G!P0E`@*'
            'v$gg0s>v0;n0s;z!&?buTk(_j`n3U7LLY2GlJfG8CD1X!^yCQuF)}f)ZRZRC$*MNgY{j268p$y@ZAvBf=EMT<55LAGz$-lHE%<k?'
            'pkK)4-%4U5??Adi#`7|cJ4j}UZU7GPD$Sw=j2_lR>L&W!N21XU+#tX=g$eaU0yhD=1H9HUKtf(EG@{XTSteJA7ZDJjP8_TzvtTG&'
            '_K|z)JbC>1;ghZ}CV;N?Ch;)(<r>u&9fH&~w?MvMh9CG>M-f!GMo{fy(bF+rA07S)YXck2>j#uH0CXA8qO49le-5sYa%vh0AVhE-'
            'L^1Jvq#24ghBgaknTvIlByhq`e>&>HqV@L(*pGb43BJo<4T0x;m53Zz8(<>|I;qZ3<rUH`6n+544Xz|qk_55%6W!1R>m^=<q>&O)'
            'E@tUXf?6(ALnTvqloTlec#**++CwfEUP)aU%@w#^rhGHGn)C{lFQ^17XQ&Z_oefdf7T_3BwFeJ#*!d|b*We3qpG5O{n4yXf*QKx4'
            'jQ*=uY$VJTKi9Z>L9G-Q1I^@%{vs?o4sWM#4!jT4)1g66`8;wj$ukg<Va_bat*svAX$hP<vK9FTPUwp80(C+NOQQme2$f--3ODTc'
            '{nZux8x8=@KOYh63Ofc@>o~olKe@vU&@xI$;|Cxs@DKDvn81Hnhf^dVe9)@(IJfxL3->NH!1tq^maN0t)%r0v+UjdTj7ILa?^sy8'
            'XG}=WBGWab*UVaOIw28W3;15GILz~}@cD6;riEJcF^_uO?MosCOa!mA{15e1Y$Fbg7pwyGpjy9XPW3{qpb6#9B{Un~Zw*;*+eaPv'
            'q1D|t_pZ7G9bUL~UVKh=bIv|80Os}TG_R&>likKK%Rat^!`5JGqKRLLCf#Zrj1YL=D$p7+h&DEHJW@Rk(hOck@^anNui+HKd*IXE'
            'Bj)vQ)o8YTMJFq^pvX4%%;{#cztV?4`L44LBnnm+tW8hLV)W=SU4VeDYY+h(Uc(w<pR5Yw$)*VNj%v>rp|MDE!|E8Q;Z*4NyB+9f'
            'v&Zks1P9CvvKJ+=O1L_Ft;y>Ou9HPp!aW`2k(dOi5X_^Go;|c`Hd#oj|4t(^H?Hx$*V&X|jio*u+_BWe$$lo5-hX$(4^z}I9n*i1'
            'y2F?hY<O26);fcWxij~E`ODF}S3_R&ILd=07guo7aJkivpydMO3j^HN*?{U0DgI!r&|4oTgZ*t7Yi*2xZq>HZO_sZa!QT234S-jj'
            '?%o}U)*wusg{y#8paf3vW0b`KY(+4OMCaB%X0$^W+b*mIu0p>-G~0g}WLI%S-~y3{IiRapgj3**Ho-cI5pLg+#gwo+-P>zDX^q$A'
            'uCvxK^|@!v6r~Jx$gbc?2An=xNO7=g%Z-n!E&sZ=F#p&(iii0ejl$yzfXkv#?>lJ^@awtgR82tEn!_$lLj0rKH~#_wZX+fkv09p8'
            'OJSDc|NhVa%}c!s7GadiFS7MDUq)+E)XIl6jE&E|8MqGv7l^p<1>`_OA|ZP$4GU)ERQD*!3t-EwaRD++)76SThlA^j=VZan#i|5B'
            'r_4jej6et6ivTuT8l>mYG8d#Q2DTR%5aij#rTkhiy6jrqGF)Qi3JtM@>h>7e2CsD~@>;XkWKVWf%Z;)<U&u5ZV2<e(NL(OY*<F#s'
            'sl5z8gz{*bW2nQTcQ&fEY_*cDauBPS!vNMf!xMv&U0sNTSsqhc*fp}A^DYs)hLT~pnuIg76XA|UAoNriz%UnCuwEiOwVEBE;eD-p'
            '&qseiSU&&?K2H;%dIj%N9tULRhs}b|+yvQ7s7J`v@%VEB7>M^kbJH6TqDWR^m9sz%cp^H_hd7_-zx_7s>Mb+0EzAU}3())RH@r$7'
            'RMHCEFC9{Uh#$lOeH#BdKe#}K(Hj~{FA0=h4=%)a0)P!Y_%A&<9A22agU{qvOv)%GdS65!h>nd_>!1`#bijv`GAout6bpT^45P)e'
            'kbg|#G8^a7VinK>7=fvpZ~5nAny$vMgdpdOIovnc?VvvZOi8C@KK@WW9rPdH35`2#>`vlaAI38VVK|JHW^;cqyxzYP9PkHDw|NFP'
            '$n5T()N4#G2_JOl{Tf2Q3kje=`gRRCo8sFrSh3Iz==9+y=?l^qyagWI;b_K~X6f34Bg?8|T>1>|5X9F-h7{71dICjo1M&Ri=eMtq'
            '&yL{}0UR9iu0fIrmmF#c!yf4%U_1+a2U}POye@OnK>#zrc$lDH3L>9$QK5<EeL#FThNSFftyf<4zdVP<zPO}bm*>2K7ni(*(nlwU'
            'AFy8`;NFJCUOdO_6(es*=|h9f$dzYaCgjh7Iwd}Q@lCMF1$%mgGtuh_<hzJA;19VVmkJdweQ~KbSdPxzwo4ue+&|L*mHE>&nZm;1'
            '4GCj=!}5EKR}BzC+(4!;Nf^#@@dn+L;Of(kO4DV4^p7Y8&5}OFiYz?^*mJn;$|M37wl6O0y9#4k;dZ=y_3N=*dfr6RMdYb=C1P;6'
            'SkfU0w_-*cRNKOvw2XnK0U3{xRK-X;VO=jLi*cS!<wjl3<zM(*uGV~U*>~3rrW&ltRcl6C9lSV!N3gXyc<A9kz_r#n#IwE%K6Lod'
            '<FV~jGy(4<c+r*2a*#}ya7BEOeGgRMLDg514X4NE&K?`pgFeorPg{7H%k*hTqap*$>qIx--eS{X|IvK=;IRAc;ZVL<x-a%knVF{k'
            '15Ro%I3Tf9$VHJFAvo9Y-T6NL7~K-F-41V=&~AtD-Clar{q|NZ1$_n{FMH{-dc2{>3^$+SdYQYVZWK}t;^kq|l$yTAX1M43=eOzv'
            'Z7+C5YNLcBHx0*W7A=7Fia`!vlg0sg%Tbg}Q3D<N`wrY&W99oX8+$w-_!79kd{3>N+8SCp&)eSOptt9@`Xp~J_8nt@InxG0SxxvG'
            '@nZ^9Y=%y0XfFp;&O$N4R)FU`*Z0HT;cyo694`5EP0>sT?0P~LeFGe=i(e7BWst3a(a4D=1p>J>=mrE09m)!U+GOs7jcrhqh<so~'
            'I9niRvjJMyg$o<bH&7Em<QpXMAot9eMFRNwzm+(ApbSG17?AW=rO-$7qQlCcKKT9N0Eu3TwEN+sgNK6HU&U03gTd2*0gmW-UvXNq'
            '6gnY63Q=GYWD)}+1hg1Gs1(0A<`{`}kQk6sa#Q#ilD#P8-4Ov^P)WETJrruM^RO^hlvXUwfJ|RSNYUoz$btxVJJbXm+bS1NB6j<E'
            'Ih`W8-tT`B(!+IGj1JG4KHkFl+FqQ`jX$2bsBJa<nrEhwNcaoH@)2=jquYur+rlu~1~~t+0jdMB>r8LqBkk*>myv%+3>kIa_Zq0('
            'PrVjeXKvUIicque&i5ykF<0UU^INz0pWkI$(V2u-fWBce6b89uuGgk0-Tkh`PzU7<MJj`vhWp-FNPIm6GLy!>0M-Q%8N{swxv`mi'
            'yqnv^KD-Z-@AU;a-dv|~xl(NP9WX#bCJ2es3wh@Ws%stQIoQ89K)dU~-CgTkcL0y-%=6FD2cQiwI?+*!Ner2w6-5#V!Z><|lX_4O'
            'h<R&d*d%AoeBs$LBv=vZ0?7_FB?}mktD7ZY#btfdfctq(JeXp7frv7*1RQU0fxOcB0sfGzFHQrW0-xsaxVb?3k6damLML~WzxL!>'
            'z!0CoJW&fouJt#|6p+`LPA*xxHy{h=dBw2XJeO3TG@zbK`|$8M+qb=7v>4sGh|+ZRq9Mt_<Gbmr#<J>L1{Ndjf?o}!<5PwTPnv<q'
            '7l9(}32J3#;RXI!MIRa|7Ix4L@n6K^qfcN4SNsS)=yP;o9|5oVc*u0}wmPfZVNeu6Ge+<(jFEVb9u5ZQ*Z{<%`Um9;HKG9iHKKjb'
            'N5GJjJkwE}CTO`H&VV-IzL1A+M6V73`c9(g?#yTH$`|j5wYIb=UumY=JE}~iQp-?6O5FzVVfhf%=*2nweL>_81j+^NsO@N?bS-It'
            '%5piXaUM>U^5K85+HyqyVdAC3Ec(v@P2#W!XK(pIr4cBBi<Hb1(XHM_^p=2KaKR)ATuZ$e?e^CoJ2H#)9SyoK-r=2=Tl(V4y8>As'
            '^j;`27$+~_ddq;@M0o`y+F2L*v?NV>AH$4DlcI_5k?{R^a`uzBRI6|)Isu5glRV7eie~8jDO`I+r^{x}%leXBA|p)@7r@1l8jy63'
            'M5e@LuL7>rA34R)u(%9o3Z%&%z)69<?d5529%Rr&V(psTrbMtn2An+2#8rZDrFg3b(gzX`+kyfrP}`(LTDpRB2Wvvg5UKqw$i)|k'
            '^9q(`sJ4Q1Ccwa_H*l!{>nUNg>`swAMJUC48~`nWLBLUYw_vuO$2JL($CdjM$7;P{HK7As`Y1NwOMS6P#}1$<92UgtUzj)S;WB`u'
            'Fyng#bqi`HAutbD1RuH=c)cxy>o8|e5cD(2<v40Lh1wAO3SluI8ZJ$JYw=CP(?5*Fpd$Ly6*`EWn~GqsN6|GVYSZ5eyYUY~aQN{p'
            'V)(^gW%){{s57lj6CkpEp}+@yh45qE0_jtM>dukv8B8WEL^4?7kyjS;-c$G}eLNp|QXy+wzVX9-CYXqvXvY~0?=oD?J!OzkW>C?='
            '!zOMWaAcpNXM9x8dvG%hh8L#QPEu=ZC0$LvXpPOWd-^<L1B})i9#Fp|#bc1$J3^a}8VgdU)KP(lyhdBBy=iaBJiBLY8O#@LvSzDS'
            'Z;yRRvTDyQjkj`|xpOChqb=tsXpRoL(i>lrMD%__Ii46l#YV|kxS=G+4*89UqOg-<ypiC-)w<aDB%TK(h9aj2ND3&KkOmJ%nU;$s'
            '*%VL`#iNc3O$1~HUyD;T4@nKcH*#HwCIZogOD(<WO9vt*>SGv<ad9CgT<l4Yp}82-tW5K3&5UjW>azx~Lz^aXP#UeJ8)&O3c!;A-'
            '(titv;>Xv=gTaCI5#&$hsA!op^5>F~hW#`8)nR4Gh{p^P5&1ovE>PSR#ua3Rv`JahLFEc@%26Wp6kz$B2`@unsBi@EHDJZEj>3BM'
            'K*ZBKj^<<7Fbu;8XMi<ykGnQ*38rq?mM<K8_EiXYG4{nZ%ATl}!#G&w*t~`b5eL)pMqcep&{cUbcRb>FfU=BBI;8r~Rqizy+0k(}'
            'O3LtVPl{CpQf;n}GYzxRQK1r^+do%5`9|HNtiJZ%JUH^qJ8&MM3Wrol7PN0zMK7mD+GQRg^8&aQO7RrtYrNwS8tS^A(Ix5tgC)!X'
            ')3a<t?Q%4|lu1-bb1>=nWvmSR#9!oY*tfk%nX5x}GEqBUE!qQdFa|n}|LYIzMN_CHIVDmPbY`TqKj`{`|CZ430S;>9<%zdbR9*qZ'
            'U=fbyo{Yl2MUcb(V7`5DOVH@I$P;mu&SE%awF`u9uaP_^H$kKhl-%VXf!D9jc69?&bN2p6SK4knT`?dlkDG@z@?p^RGj4SJVU3Z#'
            'O|Y(jU^=I&Hhg^pOSjMjiL*O?rV9OO4CrL^Q}pGpDekZw^JBH6@}?MB1QV^SNDGWdN<m#kDYP&PpTsCQJVG;j(1>gXa(yDU)v>HX'
            't=(uQd6XSuPls;(YF6lCsBU{DuPK)S77g-_;X)v#Htj3m1gfy*6Fn?Kv{rFHfjsn$f+qi(<NaM+ivN0g@`mmYj=U!WYF)0v{d7Dr'
            '=0Tiqgv=&D95=f^|KI=lzqN`7HQ_9h1?cvz0|e0_M7e|T#`jRho@w6@7$REeP={@Klt_zgUZALy&gnzsO^pUemE)lAD1gl6+l7S6'
            'mjk$%xMX*#euZGW81QrB6Ua7Ro4mYdEr0cZcYyv*=lCDYTR^xVJtHnE<h+BizYhzzXPJj~v-2${RIZLXqAtsRXq`!U6mU{~N+Hov'
            'jbRcB(|Ogdqd+;`D9orUDw<LLN03{4;e3u>K!hVg4sz4rXC&C&<x(CupYUF~L*<qI7)3^P$+E)?4A5~DVoMZyPKEHD0FCZydC2!l'
            'ncnN)X4Pz}#AIp1M`D*(8VNi^g_PZuyBq<eoE*q9qnLVfCJ?De7mzFCPhXs%23Je`@ViWOkS&E2bP`tDdM~8ZGW3{zM8Jvm$dK7d'
            'aEx)6aU!qzSm178&XY9^B918519-a?n%>ol5Q(w9O>aGQ9ERfj)<YDWj*HEj-Q3YFghTyi<f|7Y#s8482W*-#o^5^6@AuVF@#LWb'
            ')bx}{#@>9~cq<j(5y|;vm|SP*=@_!>8H`E0y~(|&K|J8^D$0uZkcEqo?aJf|xhz*fQUx=!+Y;gke5cCOq0_G{gS<!ce1N}5EB#<k'
            'hgDJh(gQUCW~`<10}vZn)Cl8wWOf-~qnp&;#sRv6=<&s?Y2LGnJo7rt0`>z9R?#NS?oSd;vD_8NqZUH<jY_XW&b~P@)+{%zKxV-#'
            'CwON_0Zsh;=U-)7nK?N@lc|-sT?3%<b`$LLCdkP$ffEw1m%$_~@@FXeDmS*vOHEcUFGY%8i8qR?a^l!IDqI}BIejIFh^~P`hz@z?'
            'eF*T{7t{5@;p0cFCBy^H;;aFTWkHgXXHm!wnymPuQ;UjqOc#SFg|-qOa5lK68<M#)lDiFf_@PyRh6n((1Z*yr8|h~iqiT$g9U^{T'
            't+ZZPz|>#PUiO|oGvLYV1J#9u$-&&VF4KX?*l5IlKu4#~U%i4cWFQLW{lC!-rg>810%Ad33foE@Ls5qmAKK5uAe$~bSw;1J%fIdX'
            'Hv6tStp45k<M6jW{`JSM!ImLNTEs?pr%W!tfo?<i-}!(4ExGvikH2F3A^+ikV)N+zK^>9Nss^gEJXw=f3QUTrGX`xwH%B0!-;Y0c'
            'Nkf`%f=wjSt8zoc!(h=59xHZ0ZW*(mq^FT1H-=~0%ksvx>HP5zzx!?0g<ZNm^dE1#AM~%zAMpjOi~+BYmg#k8U<4hov&RUpj*k9_'
            'Y<JM@vdp5sO~MSWi|#J3fad;(clVbIpj1tF-g?rNrn>T^TY~1{j7=Qg_bb%L+jgB}!d7QJ+!|6<@PRSUa#R2$Wf8&;rI#2vJ$PMm'
            'Q_8eIhSVQdk^pwax8mWG!9aXRt1F@7(<k_hRWtHaT{(E|)?cv_@|WMn2be?HZY>^^1&kWkxkL0SN}%xP8~}?NNX?VJ#U5~zV9QIm'
            'E|3^2o`MYPI{wwKE*RTMo^RN?qkLa;Ua)rabj)9;0M%sHMd;}L!+$D!k#<oh0_pJCUZWX}={aBlS;CZ3R?Q(OFodN*?|ySKU2O}}'
            'L;_h-;$;Ism^^oU`X#Fj7cr703L{9+nj;Ai&c#P<Gra+(a<K?A3%)7nH41q(-Hi6nZxyiR2K&{rwQGKBnqKz7BJFmjwB0mdQ^wV+'
            'LrU#E+q7|~G-Qs?-bsnhE_ZlOl=iEz9Mj!hKQ#T`n>BQBXdA7<(j^@;+cdw%gWn%Cs545EF!B&%xPkRZqGF?XVP*Rar0){0D@Lff'
            'k_ky$><j3TugJ`ken)}u7?#H^;S=a^2xPcKuNr)XBa)2)v%MKGN>Q9MLcsZ(#w9r@vTL3aj`{4}1kA4w?>K&4?Tdpp6~cb)GsuQs'
            '6d$I@Uo-7sAr}un@SLb4L0%tkVwsvm%87XBFkThCJN3C$$J4pW>68Md5+};KX(UaRVU>@`gI3W*N~x4|1IA4$?s&=6l0yS}YD+4-'
            '{+vVrGY4*^`G%IcdhAJj<)xKz=vzr5=|-oRXF3;q2H<m#V$nDf&qE859im)s3i3BG)D4YkKJ*AJf*s7wLaCy}4`Q%q%3Y1$HVnh`'
            'T4iLowbQ$ys}l1tyyQAxAiv@Z-+5(2Fw$R4(+u@t1`Ol)C3!T7L*pH<v65fw(6`-sRJ+{W3Wz*VLpe*oIx|DHtukocqRfcR`W9oR'
            'zTrl;D3>II&a$4n)4+X1`J!S%cogRjPGLLwd+%QHo;$uri&1M>+uGett*vfv+OXa2jqBTAZ|kfN>o6}Qy9O)pb+Mq52@nN1ytvYn'
            'o?n=fmmxo}H#tf2>M0z)ockO6E+rO_jP->fd3`o0=AJM1?%Y_6T(gGVB|35EE@MAyj9rVqSRgB8yzvE6LF?`L;DRT?;zZYPc`c=j'
            'ZXE+E+@hh)z1y4Zhufu*6zC2t?A%z19a#KAGP<;$^9Z^PNek|U<(4303NiClVIh{*x8GJG6tv-&T1eRCF<^W}2GXpG>R5RfH7j4|'
            '3~+}yYTqy)cUz(AvFlARIVPI4da!r*$<}})4p7k2zssqi?~-l|zg<>M<{6=L4sZj^;{j|@=ekR$=NeCsyw}Klo~uy*I*s5T_a+j1'
            '--zDV`v;(%DrMjzOhP&1&OK6gE8b}pU<SLrZ?v<S2a?k-Z^rMA-@G_}C)RMPL*VJY?y%V?Na#DxGj<LDE@ZV<BC+J9ZGZ7fGIzqi'
            'DmmH^=(7-W7S6~$1i0W;I#ca@V$iv<DfZ`8OWoNAd0n-P<0x0!60*N`s8f%g;Lk3~_XmT)z{VCJG?iXpI5?bbhqtCuz<5MZ=x_g)'
            'KR6sb#I$8*Fv1iNa*R59KxGQ_%sc9b^dCY-^MWN5%Dt(Ac^@6pidJp3Xi@<<NBu*P0bTKsk?=j#X?oHsAv$xymmY`+`I@3|HXtbX'
            '#VLAS%OK8VBP2LM9=<XPUXO;o_g7_HL~GJwuR?TJL)ieZ7iATgpcaMkk4nczmm5X&3q52vXc}Ie^QbSBUykYcM^2EKA#rb~e~lHc'
            'EolfZrri(6s&!Mmazh>h|8aqqQ{wXJAu(r|MiR$2q=9U^yGKH4Rb?=EpWNGx2?5ZtL2&V8RNn#Iv_}GhtI1-P$(7Il8OP+?caClB'
            '24m{1mQ_O9vwf#|TlwpH(a<qr&zMwRt}xS+Y&x)fy;P<^nyTyM1%ps4jPJYk*EIU?nA@Nx;yZERE~@$coo&e_OhNsjb?4bq7NE_@'
            'f&Rc;mIkjfgw2A1xu!4<Ycszk==JW62xn}S#y9<;3f(;6u~Q6Z&T}xYzULZ5bGu#0qgL@g$E9iXO{b-K_>#;bXh|nIWe?me9eijL'
            'MXm0vYy@l<wrET=``{x)M9MIMz0D}!U`ba6j&pQia6JS-O^fuvvk!)z-g-UXziqtv0JcVj?iQ}wAH<G^v@qT6e#=Lw4U@o@7Rm#('
            '(lp3KN7$?5{g%j;rcT|uIZHV_sQmMjkWzo>OdOpmP=xt^*2yUyjgzC+xU0r%jLR#->dv|=Q-i#V#u(-olMX`zR+S&(AX${bLcXxV'
            'I_`oVB~gLNlsL7%Ipk$pFukf$?dtpVOfO3=npT4Kc+O=?^r)wITBbyZ77508U}6Eh{wwn@oh5OS(2c7iWjj(YEmXCY>4s;Bf<}o6'
            '&~Xf~pT)G_brRXo-9jO4);7_7txV+lmhKrUGdPU72+)Hvm)@Il#$L|4k6Fcz1N%p@4V7XXj+As7$&@WX9yB@AF^4%DDOW6CVf0qu'
            'l^m;;lxcL{^PfK>VpO@|2v1<Y%(tC75@9;N>u4H|yy?26?8>Wki3bEkppge_^3=1gVPZzcnA6l3W721i_*4DrjMyO#NGY0-1CASI'
            'nAMn|vPoVCGZ}>15dVO@Ex8_#sSpH=JuP$Ez>!pBs8cXfKQJ&fSkY`pPG%Mx?MEH^Z&dw~FSYNu=f36Cgkli`E}+{G5EzoOf?Qmx'
            '19NF*N19`em|2Bn6)@+U%1j@ScQCmaq|<4cQ7k<filT&__b7tYxwO8~=ao_dWMMB#{z?M7SW^@fx=TEBzsZuO?LDmEj5-=(eL3o='
            'fftOyA@-f?u5Vq`d7n5#Lz!bD*oDi_)_LJp?--lpOwyc5_!X8ALC0t~sI#&jrE;g}(Lln<sAGMvN8}9Kd-qdIx+(JP(J!Nqft#05'
            'bSN)e_nA$tv~l37i~;?lIG6@kePf<2dRemBx1>aCSFBX0npMH$Pt*lBQiYZ;)RZ{($LqQxS3yJ>S}cF}-G;%&Pa}Yu#L}_v5t)Bn'
            'iZJ-+x!2+1HR!aHL}HU(bq(o4@`Y%X7>Oqq__4fo;84-m&ZA)j9CX#=g84W+I~fMobq2oX`H`WiT6T@NG(=oVPd#dy<^wI^7(;Lb'
            '9VxC_D^?x*2-tf45q<bOIP`<A^R9*IwWEr0j((L=yW&;N5$Y}%?y=^J)@wW}sCD>eD2`Y<d#|omEO~|zmsrY#-2s{SlBX)RNYXM9'
            'WnNMu>;+~MxyjOGfv`IJs_xJ}vf%sg!X$#<s6Z&Qc`)U~A%csAHD{cjz_97M=mS86c^U%U3S+okao*5k87-E5JzlQc5*p@55Q2My'
            '{!@yWmD#aim<7V3fyyjQs5+Py^nDoyF4HWb$6BZ(FexuLd3LeWHjf>Fv8@alt0{TVl*v2}7K?D!r%VO-2m2+=vfw6{xlOQhm;v7u'
            'ZbF%JWr1<42r!X#?BFp5y8r{#BZ!oQAjNM5aPeZe6Y|95aW`vdgP@R0TvJpOU#5ChrGcMx>;1jgC-2A4Pu{$I_2T%=^W!ll_s5j~'
            '$EQD?ynaE3@b4cJlCb^<ePvUia|{O#3Ppv9HRky$=?Ebkg<cStz9x3wb!puZnpBDtb4gCD<S?f)#mtC#M%)WIkyli3F2|t1$3uL3'
            '^zQ7Gy{(XILwjfD{rApq@4xMK`rrPsOTYdf50!-dU*5cWe)8fNgaY{E#mU*x>(@%Q8Oslk`7FV}J2XA%Wv~KK3O>(v-O7VT%K%5~'
            'HTf+YvyoXV2nvR!hZuvcreG|E<CWZz9prdsa9(~6WZrd5OcvzG&5*8~bL=G5TH&ytNnh~Lb$=ueT)?9Ve0Aj;9F}pEJ^a8rDgsPR'
            'E7kAHou^RsQOr4_H!wO63*?k%wuM~z!5RD1r)MNK(=@wQa`&1q9cMh`^{7mJh-wyYkv6L4wq;I#q&AT5L+skeg+z9=(=|$ySPN!G'
            'Qe&69M^-6;ag5tr1K{?ShPnkrB%3Z{U4UpauIjpttF_HLQs6{|2I(iN-&LaqRD2nIz$3p5G4&<i&hKBH{dDrn84S3T3H+B~l%X~j'
            '{>V))4wa(`NMNXHV;G+!cA6wt4yv_y`z#E5m9n*RGVT#>BBhSW2{DrSHWLbTKo$?*)2;ac`6d#DL^>hg=~?BaP8p46LbH|VFN1mv'
            'IU!LkBeIrj3_Fxm96g$D;NPj-rvM~Sj@`xdZ7SOWBDBu)YU-#LWk_;1eh@d5{AOUxO4eq-Y^GaRVYtp^&Oj~GnIZhNE+pt@Rq{if'
            '!a-93x#(zm#4XgD9x`L@6Y4~v_M5wue_yJrPjAAKN}qvXK(dc<c;RZX2^Pj|r~)zS{&Dj*kS>G44Vo|QV2YfiOAOXQlsS4gXby{n'
            '@v$a`A*Xm@=-YIzYa|*S?gMQ!Z;Pnmbv3`~*Z^dzw$|QQsi>_l+S2*sWI!WGuG<_;N@>|T+}M{bZPG?9YXk(3b4N>+Fm0b75o4*I'
            'sK{_h!O*FQzhscGp_(Ykkk&W|HSz0PiLNEfr%gZxR{yclRjZW@oV0gSsr9Y-7;Z9{{CC79Yi!NFbKl{GQNN3^yWBD4l&Q9XzLmRn'
            '{V0axntZab^(w;8xWKzz?Hp|^kpmN#@*F{5kYND-)EA5|1`Xy0n1)i;qLl}y-q1iVblnHY*#m`q*stsEQr(;;wcI9xHE+?>j3cLE'
            '=F084VJEsZQ<XDOE!Q-&S=CNrXfA~rr6l6;m%9zKH;oI%<Rukk{R@$Xp~h!C?CPMP5g+#p4HDJ9)s6&i;&!i=wYc?+S*e=uOTrZg'
            'YOn#apb;h*07<DZbj^5sP#t9u-cVRj#hQRO3yd4kS86JGue@4G`W*)!BU!UZs*!sI-1X407;1-9>CX&w3*&k;rzrKY`j%bN`g1Z$'
            'iaShFH8_m<QMKcm0=%M}^0b5aq9^GyZ(Spj+OMRp3GVB{+Hy(of*q?rTtNCoUyf{y^{vp9(J@NvdYD#9F8kiSa%PAgI@(_0%}-@-'
            'S>26cqrC|*VajpboswzhU!_z>+rR)dhITISTVG-!izs)cxYn7jl{#2{6FzyR|H#9b?_pWq;J#2>%rd^CVjZ(r@ja3x3g^TlYrq-3'
            'V2d;P3nl2hdUH>R;Vh#14Xn!?N3&1suC(2%yxmCW3$y&c&$NWkA(hpsfs=aj=2)&c=G0O-qe%TH&+7|Q$$_!56qP<hRRuvo1%Whf'
            '++-;w(m>W7y#}zXy~@z7q8}6=6H_&Hkp~+x#E`j3`}tF9bIkjS=9vgnQCAtVNOwV!-kw@RY{I-Ai4F7U0|)LABf#?KG+iWEXIjf6'
            'jtliY2_(90?Glu1U8FA+U8a;I^YTdbR$$7k6ml6+$V-+^0!%DgY}ifv5?SPetqo`!FlHnH0uIbE10*UefcC+;$0$m=zr-vpCj}NK'
            '2s4rx-mo@64Eh6hZO4kC3(oQ_!6^V(%tb~+W>+!~xLzBcL!939s;`q83#;n&cpDTvR`{#}SyRryLJFt^Zs_#5FY9%XHa;%VK10M6'
            '$Jp0(F?eAjW++(qe8X&jv24HzFfUkR1e*^RTn5HP&;-xNScO75e|1~HaO8J!H!LJPQUeU@H#T<9)Z*SMZS0M~=TI3H6j8~UQNLqd'
            'uW1^yzNy<;1KHaCFzt)=`Rns(OJa4~wYh%~JGF#+mEKwUXKBWd`0N_rSS^4<3v6ku(UYs`3)AK=OzkfqeXu|9jX1LimP~cX@xFeQ'
            'i}3{J%xlXg?NFu?)CFZEG>WI95MybZ)jS12zJwd>Qb$>V;LX~CER;6?l<M3hrQ~3~VQP!Y(Mc$am>UTcQHoEnz#|p{VQLPKi27q3'
            'Pk?GhRL&dpHw$i%q~@7}VDLmTVH^Q!)Hqe;zRy{JYdhvfFuaGWtb0t2jWDGTvx92`xeH~iFW1F)ou4?nKaQ9~(-6(V+LC*MVQNOo'
            'Sfm;kgan}OY*6)Z3Nx9jr;-y<ASlM$pf;(Wq&Jl0=p(ZBWijpcqdaAUpEXcKWz2G4k`!qL#4z=NQ57kb%p;A1W~EFLR4;X#PnY2;'
            'FbdS0CD|2<>v-K7SL>O%65h}e!n`^?X3UW_G-VjWs4@vTz)R*J<1Bt`$JjG{2&f8+v}aU<E`p+@a`w!SjWOK<rmHfmkaMG|>~6;c'
            'zU;4@%M(laCZ_2eHl~218lek89-s+i8e-ixv`tgmam+R0?O-V3+-c+`lnc!`mij;5q?8OJuzFW%QsygIDaxUS)FDO_^D|)RiVcW&'
            'a(FPFkLJzf0SO(KYpxXwGqk-*cA(;Jq&LI}XkZEzwKlGFQMGTEJ3ix3HOL$u3mu(?SLO;yr3C8*(DduxN;Y|GX-e0N(Gr7YV6@?M'
            'sfOVy((uep+ioiIt~5E33lq3x8CjoVvmVlOJug=si-YTm2Q*^)p)VeF?<WFr1}1|e)ae2&QfV<*V|9VpkI#pX2N&&~%=NWjSvyfN'
            '`p{O&41WFYa>b3>Rpp2uwVMh5V=jG;#rEJbwdk&-qnPr5No>Pg+#LD#ymxSMUcI2ibc#09x(>{#)@4^ZuvCaX2MnpdSC^SddEeKW'
            'gFsaPvIRG39ye$*vmpjBjGg5itT9|t*-%ClV?ZyE=!l}mF=Zo{gThxgC{Us0Wz8-C1~8d-SxTK5Q~OyoLlM}Z=&lS7i{L&YEx#RJ'
            'ibK*UHSqaCuIR)0g?=K(#t*9?Qu9co{HXOwl&L}8VMZ*WdNiJFsshydAzkcuY95ek6LPxUl@3djQcfsk6?=Cac4r$92cxMfjGlMy'
            '<E?uyn(zf4z4!I&-TM^n)@S(seu)P#TkWLgLpyScbzI{4n{lQHqUA{A<u+)WBkNqzzF3vEU>P6lnUSe9Z#JcpP4dM00Q)-@6>Ahw'
            'IsqAAk5w#;42Bex&ngjRRArwZofpGa*!&0b#r>xT?tgp_+>4r%qVY&Zmod&eKDSOmbor_t%c4RO={-C7cD19Q{Ss5$+(Cl0Kd|i5'
            '5HDsKT2;GdgR2`Tr|g{6y~X0)5IHC07kPH3TMRu7Wj04E2iKjstoaa<OEf#mN}ZGbPS-ad;o3Eg5&QmO#E_c5E+emK=-;8{^=Z8r'
            'xoJkdZG@utTQbx<@`u${^K)|Qe%P|h77ytv3Aw*9W8*b0H9G7fe_jW@?fyHuhf90%XSs*F!)%EDlj;U!#H6dL<4`W)8&0cES4;oz'
            '_9nNbFZ4;rn|2jliK{S%b-cqTeO7)Hr|-_ho$lx|0!BHyw1!z(tJTW8HLSjkH}xxbv`d~g`6}ZsbFM7=&x&0dsUJ<my0+uI2Ff4~'
            'ciBC|#bpd8mf~_E$9*G)+m0aT`0ehUk>qvQ98_KlD{ts)#g?lM82IfAXR!4S2daJLRY-%}4sYcrWiUCyqo4g-6DU3M%(dj&*B#B^'
            '+iR%Of1__E#`L<it^eQu^S?af;VnMvm`~8R(d(K|e#4Z|p3nh=<WZx<50=7B`?eX&E+0!NqW$Kq#l1fq>`K>l*nip%Ym$Y*)p!xC'
            '-3(w)9#^sS)j+F=`aj{kv^|yYV^T)u%QT}5dz>(Dy@Y$R%CoBkfN6kNkYcjgi4{*oma*?Do{t7kD~E)lnuxo&r9yyHrKD=E#pK-b'
            '(xLDIbOAyQ6a&IIWeT}ajT}Q>EgXg0a=@zS$%hiy`QV}odbPm7iaQWgvl<N2mN0y$VuR$s;a!fIL<(uvvpUK|0hy}cfsN6jvD5MT'
            'h^fEBfBg~vHN<~C^BjvKXLcVGo><j}HJwC7Ab+td!^t}bOoB90cN<L<)YIB2%$|e&pH}?|(VP40t@uG^%y_I=DEBZl_s2c>L1o)G'
            'P<^cv(cH0l<*k;r+9>8a))_ZG09AN1FK0t=e04tdoQL%s0!^o5$A=^_<Q7(>ISb;fZv~>-M8Eh_##(f7uo+`o)enj~VKv@eVu0?k'
            '(LH{Dh{;cYjUJ;QrqvHG!oLQ;mf{q6g`LU~>d6Wf2KfCU63A>3CF5)L?C@~ss2T`zBjMcvLN41hje(_sb<^|sm!I$|eF^{UoCcLK'
            'Z0`N?<(<7t*y0S``d9(4@-OY`5{Ct9Hu5$3dz-gGL$=5Fh#KPP^b~EmD1EYoOkOi2BQtKnU-_DXW|)TOViU$x9Et!P2bg;o0iYLa'
            '7+^swkhSO%u`UZzka8WlTo#F^6!^G4Afd~$%F5{+VUSe8F8;xFih5z}wy(|8d}jgK-BjA8aJDbP`5dHnE{!*@bXiwH7Qj425iD{V'
            '%z~9v;(4%sUKTw{roT!l+(l*@rx94%BtBeN(W#9sFj#?+{$-%vs3MA{p8x}Pb3z;SgS{7$kH;O0V1E@u|1m(npjoh5AWy$ZuT%o)'
            'C71Ft-9)_eQ&x$;O6KW>X-nkim_BPLt{?RWgYS&DbMGyj@pobV--d$n098byyKQRs7q{KUY6oA#g5W>G(uCPjGCN%cv-D;Nm}Sxh'
            '?|l}mPcebdP#l`i&EMz0KKf2jquiR$OR_Hy#T%eMzIX`)CDFfFWA`?TfTV1m6aM+)lsHi1!AV&lf~r3&!Oi;-#1{4V$20~B`V~eW'
            'O?~kTvccF3;K6-;6`r6g2F>=r$^ZTZ{>$gn7e|<P-UdxeLi5ohfDW`i&|XP`zWAA_;cNBdo%->sFOFp@3~OWLIvx>XzlX2*x0*h0'
            'ecCKdb-VDurd{yBC5BtyGr%)N#DDyM`RV(o@@MbBAJG4STtNT3<ty5zQ{CZWi*5YJ@V`R~EL=SB|D26Vl7AT7K`SyG6D4azgxA3a'
            'C6rTEgdC&ZLoPF;$gTw?g5rxaa=$Pfh9u#%+|`*dpgi=!=RfSYS3c-nyOx}6Zrsxiog77(%7Md6YIJSkSfe?A1|9sxABcHt0?WxS'
            'J``h`+}QpKbvV~x+L&V?OmlM~+)w78{LTE_0e8{Pn%g-Icfhx@#VcCuHZyYh)qyIH5I|^A+@!X21Xmk*gH*A;a$^Daui+|hfVP6W'
            'jO)^`MNSu5ym=;&-{%;(zoL)3QPd7c%UIX|4CM0kT3)vE02v;2b@sUU%ye3=%9vU3JfyH#0dy^x^!J*Tl30+g0o|nR#L7y)zJxt|'
            'aqyYy&_-o!^A?+W+4ccC`a4c$J#Nspo3g7q(Dyr<IthTI$ksWJ`o!FIsDy#$^VoJ?Yg^Ys2XjdyMwX65f3b0;t65+&Q2Nol;P#Rz'
            'Nu!nPoO`#ozx_6?-w*%t+i#PD?e<m=ckA5=#JP1Oh~|FvSvF%u`<(R19Qr}1>na{>5Dh)rmukNMA`0SkfhK0S&0HNJ84tF#=V(FL'
            '`uh#gj7wbJul{W{=<R-kAbVFDEy*i>C#QmEg*9RIyjsm(c9I`9oL?lRM}q^6&3q@nVqDYu6excq{Qx1@41+dQIedieNWe}8oK0LO'
            'RnwY74D!ZMnmC&DxuvuzQLrf04Q7$p%O+lz%6-{awoYk*#<xx!tR}Mn#>GW!pFDp2@CkY<dv2}NrOHIk*)q)4T=tjY2N^}xJs%!C'
            'xlrZgWr7!(DU}t@Xj+spIQbaM32~f1W(tkjA&6h<$J9$^PR6N?@uJD#&nEAHw-F}&XWthpVTOvqFuY<Fp(?6{$CYfUL#k*lGtvtM'
            '*w-nB(c{SYw9gNNB#1Y8l#daF`GOL^>R6MTWKXNh!W(L5l;xOWgmfZ%8;OfoRO=DPtP=OpZ>3f8W!hhM5yI>ax-iE}nD3jkc!~OV'
            'U5qgy3!+RWpmbh2yL$?m9e5VyR~&wUMP&J^#&G303^Hc!(u9<#A!jINhwMOxA)7@SnVSqISu~|;bCsQ6t6b%NVlgR1Wd|9gr$75g'
            'b~$`l`B57ifp=2{hdbFV<bkhmRa4h`tEw203`3x#k`{uRLmgM)3Nnx_ncbt8Y^BD^I_>eD->fxs0Nn9$r<v?brJ;jK7^@1*Kq4vu'
            'ykVk#wKy0W;m*ldYhX~zrlI@wc@sEyl)gJX3HM^?nmY|Idzy(8x6x#*HS`tpeUPZ5I6_rRAiP^zldXr5T2UUvaP$^YJQTMC`ghs3'
            'Z{lztt60K4@UgrS$dCe@tiNw5Ogd_JelXT=hyG-~Z!av`Jo(qQD{7m9&eNS+{r4723vOUAG-`JwF24NGuoXK;<^t&4I#1yH^&RIi'
            ')}GhLx@TnuXu1MOA4WVvdz)@mo5Rc)-%foi3ef<KDxlv;i#Cdj4q?&*vY&s>Al$ph`EUbi(;u(D`<wwZ-E%grG=|eiYRtRO>l(Xd'
            'H$Jx;UVPS;?FLDHAoRd%C?LSNUAsVSme_k!=LeEzZXm%rJIGt4N%Rq;1H=_DY!ThGg{m7o!LWwDu%!fV2eRYA$gjdAN+N+drbMs_'
            'K;-UwhPl7Wuj4Gxi1Eir_R+>-C@&s7l$wfT%4_w|C;_3_5Gwqj!#J*jO%hzmdXnx@0OD!v<yM?aZw!NM^)!vkFU+&nFew`!?>D^~'
            '?WtWZFx^_CmTB8a0z?r86n&H%-f6;)j{%av+Zr1}O^i!@n|m{8cok*qC|*T5teh<)f%s{!q%w;)P}g3KHmjwrq%HeU>{>yp67czJ'
            'HuvghqhlUewMf7_;sLAH01)zuIr_hx{Mkh%&no!$xg54R)`ol$WsbaXr+U&=<-e*7Wu_ND>c+^ixJvmzwkj@)C&QzatGPJ)>FA7O'
            '%XuzG3AzZYx2Y`TYQo1%Qi~+TI7QR>w8(&w$1pDnjRaX)Tb8PXcyOe{HRquU3Y~@sB*E!Y>fbo-tA}n1Qbj0p#7SO$6;9F_#xz{Z'
            'o}6E4cNPjWbw+a@iYm>Q;#I0XvK);a70|UZ4e9uRU654*RutM@=p4pEK;2gqztUA+Hd&d}6H34`b&NWg&El>bjisJv>^C@`MMj<p'
            'ow4~IIk|OP+d-eI5*4Y}0CZPGf!g;mV8f~%)IEfBU%O7rloGH<3pkc}TUH0{nn##;di?tM`PuP{wqy<@!0=&W<Zei{M(8e6Z(U&;'
            '0$5~NeWuNMh^YxKjZ6}K@eAxl3<!9m69*Uv?}js<$*f{Y9?8gmdB~C3k?9|E%tuG**km5as$7+f`q4?`%v_&sSv}<mt+E-+G1>qh'
            'oRaxkRgR}qk9FBuAEIatmjmS7nK~KTD<|3Kv@CK%pTRI`AC7)LW&&)?l}cSJzx2Y$(QbF+s01pICtVj|@+x_TPSUA}VXhLQ87=5*'
            'H&Gl@a!*uRa4$}WF^lH&kQ`O4!zc-&Tr5frfe_(1=T%im&R)mJ78IFE=JT8~xhIsR4mT{TB}>k*`edA9(vXc!`8bgq=O!jhm*IT!'
            'w^3|~v`6DNWgNeiuccPcPQAnDOR*GEL!rYtWwz9j)2y!xeYzy4V}y&?Z~PkB!W)z#dB^garZ^2PPr9SnB{q6LGfDw_5UnEEhAD<P'
            'f5eK=4wR}Uv|{ct*p*^V?D0yiiY)?goUC*ll~b8)6o^|TCs3;k6&_1uME-<Z&s~Od&v@cpZ0-500n&Y!xF6lQCZu!-DW~$)=+*+a'
            't!o(2j{>O8Ce^;p&fN7H>93z#FlC-Q<FZlJF^xxEw@rU_1^@0~QUrjk3bLarahzTm2*HSt;RTD3gh1%QNwGl;)4QKXz&Cu1GW65N'
            '7;cI2pP4V*;`Z+5@#KwVv!%@n>(;BhrnBCqtv1vd?U%X2oYP-^JbQKa`gr{Qr&p(MkKc`dK6!Ec`g8{dIdZ`a<LEIueid#2t8{G('
            'WznTiQW5s{!Tc`UYO~3|l?ZUAc?xM9tnnHY?Ly3fj(F+E=?&TpE6KDDOmP%Lv$`OMF!6_)^~45Ot^?=Y6qDBHW!ZS#WNR+^Yckx)'
            '6gm5IAj_P<F!RoX-p7mE!>#8x%){a~>mR9iXw^g*8@pd$V_qw99c}^<UEwLp<}oT4K=1~3M5Hj>I8q8UV%&77uIZ9Y89J(C)Ae~z'
            'YbWEbU@Gp`r2jF}QK)^9hG96^WS^-HoyBN&tiODF5?!h~irr;Sd+WAW0d9;qvY|Vis$D%NDU%b!USLwm2g1ZUY}H?_A9*I(!zkj8'
            'Cg_qL1G^c;2AM;_feD#at-x@3>*cv8cSMshv?=x?`Pkk%uUDAO6{Rz6mAzH4MhZL_UJnQOqk0ew2Y)oZk9yPLHNF9k@WbJgN7R_o'
            'YEvE;%%Ou^5Ivq|bNuh&Q*IP}7|&PqW>a22IvlJmu)&nta>rqS{|$nvnvhmjO4Yt0Bk<b#GWx{yu^!p;-D|jkByEGw%-u+(T?BB8'
            'nczTft#<7j@h2(6$e{a0isbso<CiDzj#Vr_F(l%-%#<J#CUbft5zL|A4SLzAtY)ML;Vk+A;FsogC9f>A$Sp983zO(Sqd0#Bbrxdj'
            '*-SCY-*ET5NY^SQQYB4HQ_Mh(17S!irgR=<dBGW#`wcv&657y|tONgHaAh3sDL)q`EY910kZ0<|cGNqcD;ugvrltc~`H*wpb2FL1'
            'g-K$o(>C9~ZEU@T3;APQUg2&buStuF_TNPutk6P}p_UTf#j{8nN36KEwm7j`bbqC#uGD<@CW-arNA03{)uth|YzHn^{d9a;?T)Gi'
            'nl){YzgLSE)=RIRco%YJjfWBaIPl<~RRNHgO2?#*D!q}8fZUO^vpFLn@7ze`j@>b7R|p~w++vpfRyj##gcvyzm&vWzxg$C*gzlGb'
            'XhWi%k)zczPFK~hmR<AKPTSFVqIR(Yc9IXPyL@_|?o~z*2ZDOB8;HWeabB)D53kI>B*S;DybAiX_Dl5Yzfx7+Y;i_LuAZC{M`i@9'
            '2DzEBy-!P|bL^g*DUL3xOt<dy61H#Tu~U3gV8sfqKg3)la?LQd(xu)P`@4cu%<hfJe*n;|%z;`i%Igr;V}Ey6tqPivb?Jh28J<d<'
            'MSlI3yO8Q;Xi$|VPt`eBSg&d;f4(-=Y)@3V{U|Ft0f~O7eO15l%xN{|B*yTP85N#dP^hc1XPlv$3z%C`Na_ujFt@^u_N=hF@J4=4'
            'hR(2vtrui2qX2X)(V+3pumg^MKk)CT>-`*cbG^@9YG=jS=j_?98X2Kp9H`UKrAY^^Upir1EeUC?*40wp(YiUJ>W)rLVN`c?ae}|>'
            ')>bPc9PCb*COSAEYxgQ2oHm&E)ALhy<MGEdT~(o62YmxoxxqZ0mK3g6g?l~rt#=#yopDrlS`OZ^nwT{H@*Y>5oT<&m+CV4oUj6CS'
            'o1@p#hJdPRodcl3fB@WjnnWNK2n-$XDf&cFh@?5xQCl<VIZIbej)|h#)eg&cQIQ)sJ?J3t`t{GqE1jL4Y7H}&v2sUFsb8=t5%BNC'
            'vinN9PiQ8T#+W7%<p#S-IZn0u$4HpOsbbKJ{yG$fRYEJoNw%e-IEEJ3=n%MP4xg89`nA_zRTKV?pbF>h_$-C^{{0o=`~OK4;*L+n'
            'XKTk@hl%~4!)rW2*b}5bTw>1X=M1fDc%y%jf?cx{TEDKJbe$GoMZ5l|2q;ht)Ncl-r~DhJ)<4CO;@@4#ZXUvRz2}~(xBNr}=%=VU'
            '@6k)<z_oiw6{*GbU^HqO7oVpuwZK)H*QO7+s?~oid&M)yn{MuLj$o`Us7`vzSi4)-8>-UR*H!Re!~D?zwcQTle)?}=5NVob{rjeg'
            'q>=F-Ypt2%KdF7;nudvMQ~wwXlWMPVZN}V)D(rrmnWi?-CVNfO8tedZpH=WwfQfe>dylnb_c#7b;D%9$#W4ic?!Zo(CE?W&Gpga0'
            'QNaCx39W(XC!^ci7vWgBci|3ui$i5T#qMJ5gO=tDI#oTxI41i;=bnD+T{M7_#f~u|yCwp*gQFDK7tYX3VbB<zD<d{SbsM>3tz<?6'
            'vMoq9!nK>YP_<a;tU&d=38sEqTLCSAHo)WOZSd%A13VT{YvAPLW@!Af2CRxp9mx#fO(<E#?a88|T?dREqo?C@8-Y1tem*5Gv2R^M'
            'CrlWM2PLFJ&-M*YEjmqCvZ<6p`y(lVyvm=zQXEya#xVerM3g)%eD&uvo9o{>rgcF21mDO4Y0txyULb2XP^wP=n3MWO96a?=?l;+7'
            'QjN)cafh62$yAz&Z!K6|@!)|tgolR~ugqGsMkw-CJD9yH!HsF*wH8>v{-<e|O}m4;O}pQSQ#b-)ie)7p_YaiTA`5%G6O{1$=;*mD'
            '+L}inD5*{YI3^|_ki9-RJw7$-OBA_?gAH0M(LtUi0aYQ3(gz?ghOM*XE(89RP`$m}07nWR@0Uay1=QEoNA{oS@$7qjafDXEf;2ts'
            'Q;ErZ40x1XPE_UeP{~5`*P0?W|Ei*eR&6i-q?y;9iqc>uDyduHd129;I0`1Z&}-grGNoHv>zLGSO#dRgx;{9V_t_VGE1?>k(-*Kf'
            'iN!I3PK%)FI|ol@gA8)V06?Kfn~qc@vDY;$$CQhmbS0Y%4L!~*%xe}U)$mloD13oO4YTr_?&S4vx*VCEzEz!g8P{?JGbm6+2S{-h'
            'arUBTLPfvy=((&9g0^sWBM4T($G#%lRu)m@lt0y0+@J~2ic3U+6Q`nz%BCvErTrO>5^?nQ?Yom-k6&26%~3*1OWVnrRB)tLMJ9}#'
            'fR%%<1V=AOB=fbuv5X4ff>+T8S!0d)j+WYWzACCl)s}i%oryOwW&2`pU0ny+C^~JI(8l7^y`BMW4G$1$0>P;}HPg&xq_C8nj$qLF'
            '|BZzi)m7$*R1E<lwRYXAc!x48QVF=RE^mT*Zu?bb=F{(@dDFlu`~}R?%pTPnYG_@Haxg2QApn_vcz1seg>bt!j~y$cSYimVW>qy{'
            '-HX#4YgAT6sAT>UJ-LSOnakMxIcRaOugTKAY9YGy+~zc%#)0jLmJHwbnCpd`{`nk7(lDa7EaG=h@S-lUR^s}6U+s0zz^vO=1Tf?D'
            'y)wB<;0E)y``s2Fos%TkfH-O-zE)Y9k$V!ITTHRRyL*>Od%jPN+fmb~Q=>2xt=k{NxLAeVeJe+H9fRCt*_NkKz4;dCUJ+%oLKP<L'
            '%c4yjC}KOqpo<6|8*WB6Z*9A-)Hdec-bZ!jwwAE^KCVSy;*HYmVpP*A?5xSK+wVvvJBTa1CT9=iXUJig&Uyn`P=y++GhH2<T)%At'
            'yqiOC?^|{M<H6N6!e&ulU%dYsW~pVTW7@I`1G1@MCJq)aRL#@`SWNP<V56L}J*tbhz@QkGW2mAm)+!{JF`C#oi3b+NU{+R;B~0?R'
            '&=gtl>K2DvJDP`B_?VmsgG5H{7?Fe1WjMW}qA8Rx1g)7lT&;QP>uMGiJlsKCG73JVna!vO>b_%HeEa&BQ?gMSR@i;|GS|cn;<<!K'
            'kYr^g)C^yv$&zpFB$d7>W5b7Br@`0TY;$0DZT_C^77=6#D+TmrPMG9=c-Y`kp}RWD(I4+qb&mJz==IrOj{f!4>8pR$)$wX!<fni!'
            '*8%W3047g??1^X^X21Xft13FH@?&)MAh<%`)&w?gnP4=A%2?5@K|5k>mU>1d5fw<bk>W}F<?)AgoMvJ6n>P#e9PqPn9vpxO286l|'
            'a$#nAev{O?Yu2?k0d^-Vn&tP(f)7pj!<;+{{_NN^wt~{FdvbAxX{y&|{A7PE#YgXS*z#_5zpbVSbX7!{rNgH4+wL7z3=u!;IJTot'
            '%@d0vw9UGOh*cEKO7VzN;vFxtV7=6?m-;c+=b@LKJv&30GzFSU9^cm1_yYYwSRXU097GJC5cUQcS-v5@e7vz9Z(K~k1Z2B0LEV{s'
            ')Q++zhJA^ughb|nzWXN7Td0<7|C|OoyGVeuIll%m|6;%1nu~~{rVP_FRokw{M4YqE4XiRKSJn>8g^I=Dy89GYfwS}0w@kV@+xA|q'
            'bc^4x=Jo-o66}sly`GVtdo!|Bb^cy0fi~*!M|DGhU!~U`)ph$e9Hkr{)l|%8n&Y~D+=ZfYAJ;YCc1u^SKWa1<_<QYHwnq&n0K2``'
            'SdZ$eZioG{)X}3Bm3CuAr*gu(Dswa9)ycp3DRQ4o20C?2=<b=@m`kUJx&oXzblt{l=r>N=vTF>f4Kx{fw=E;nP`X=hAJ1_$9Hxj~'
            '^UUp&Jl97VcgMS}EzQ=ub;r9M=F4D>YmP>`dmMjFjdq@Xbp2?`&+$2%WnXt1jHD#)a(kF|`V>#xZOsRD>sy(qM^$=w-%3F}GIA`t'
            ')_Egs^ys1LHahfW3S;A|>ilj$ar5-do3{|qi4$ny2xJTTW9!{x`!suTPvXtHF~%x;G3GPHvKP`P#lC!fXG#}JZAc1BjwNco@wf%0'
            'v9DP#q@IwqvlI;VBDxAO!~84@{wluH%s~-ep%)r(cp;@5GYmh#rAX0<{qT@m$Iu{309vKYqnewT=anjH<G(uhHjYFsx@&l4FCb3A'
            'deC4H;a#j*MDA@6ajag-CW7OuEMVPFiF|8Q*TVqQ>v<e3NPqhs8euq1G=HtO^<S*amxzO<WRvMeW(+hpOyzu{&1?$gk?xez_3U|c'
            'RS@nZ(Upp=YVbmnpRM|$o2K&1<d+o{YPBTPmSZ>fS-E!t5wR8q9g}NCjIh{&ZEQp9p)U>|_f?JHmdTu}Md7Nm7wAHBuY&Sz;ABO0'
            '1XQc3&i+KTE^8`GsY5Z)76yOr{(!2+eqmh#p4?54A&M-G(XViUYPDxFwHGEek~E{SI=fw){T`#;vm``va+w9o0HpPgaiDOEzWo5L'
            'EOK0>Q4%hVXQj7iBtx@NR;e}+VWpqAdeoSXKxG&j>msD6m%ZIlZ`~+P+~SqXJ1Xs1s<l-Dd%7eqb7P&xa+!v^mQW=UW(yPUx;^Z3'
            'a-KnbGBeaEB?4}kVjqiB<(%gm#m@BV!nl8U0$&Fqom%P<tG%o9U)5=p2?eWKj*e--*1luafGXou3jilGs8J!It<<}9I;xuQ^;xLG'
            '?+tIP>hCJUc}tadgVZz>c^?yPY8jI`>OY`Q$75P9^RZktXMB!E;cL#LB?-4Y5km{VeD}Q~QA3AI?NoUOnI-^_F)N>mqFmWZ7r2t5'
            'nvQ;fN$3FYeMdzlyhe%^CN@nIcjyXis<vaiAg)j0!kidtz(1e7`Q`NIBg}NCRDW_lNa$Q`k*9QlzPSI$oAs6rgBHz$t}XyrTqevw'
            'vKQ&9jByKg4WwHu^KtLX!5WWcZkP4F)lDJUk5zys7>uhRg;ym(7&r!G4&-Sd1T!5mYP^jS;4=_IojUh!e4#h2dFL9uTKlD8^lg0S'
            '0%(lv7>bs#Wizp;<G)WAitW+g*v;^U$M4i0R=u{yvg!PMwS}wh8hF=m139Cp-DXHE+-J>WAey<Tk3bnv)G~N2QK)XoH_uRcR@ZUE'
            '%2irUo%p|c)NO1m)t*j!`E*y8O=Fjh+r@MhgdXOfPD@jJhozKFWnN=TX0)rOP3k?*WYMd&Yl3RA?^V-As5|VEh^zOtT6%RKZteL^'
            'BWU#kzXu#v!Ku@B{aH|?xqaB`Y~BW;Awz&FS8+#<06sycx*6m0&A(gPDhJkT{hFgqs#T@uZg77Mo?gc`aB_9;4UGw(W-WGrwOa0m'
            'EWLx#c9KUHT<yQ?HfaM`xwQ`)yyJIulK6j@uH5b<FQQ#qTba#!=cg5>w!u-JBP~E`Zt_EM4~@Fl>@aV3Zd0y%HJ5m^SBrhXtNFdV'
            'of=F9p6lIi_mNhL<}PNjnY)Ytx>KzN&7JlD#a@}WAt}JJ>jvJ4FM&em{b;E6=~-GdXk%;D8r!0tZD@L#H(u`5C0)DU>(sJ|NPZ*U'
            '|McqlPp0o9UWz%V4UKZC=s-UMG=~=C1KAhvISVqmHSW3P%Q*e3%GoT-o&`FCGO(b9^w)^f^eSiV*N~pWJ~-ge=fW7=^Vi2m?@q}*'
            'T5(NQ+UF=y>@#L~BG2RofVwNYOy{BWFDcN0zg$a?THUHh1!-8JtphBW;3`zUAvAUEmdU=}GCIBQV2qJP%}S9|lkZ4os(W3U37>5U'
            '<Dig=M6a=ovgxcT>lW3WHC@tV^ic?cBi%hN-LgjLlv9!{EXBSBb{iwb%TQFVaqe-c`8naSsXc7#WqqXKsnf4b%ta=?<(12nWQ8g1'
            'G+3tGPv&`Z2W@vWgH)@>RSk(s4dHUoq<ba8Mb<KjG(*?c<8*7b^gy@f7w&uvGdtZxD0?WIJqAHG+mYJl&}7m#hGIT;2hyVB!o6&H'
            '<v?gvNg=|vy{k!f$`7k@b{{%Uiw@B_s$ApHr42q+a+zi(sz?lws%eKgbZ}dhITpH|1Vs_ou7CiuNiQ&KA9~i&h1Uy{Fk5VZmMsz@'
            'u?Z0ByreYRH&HT6Z@75#hGIXWqIL;z4t*`f6RAg<Oy!z`WFt}{+PWr}?@-Y?)lQTnKKkz&nZDng7&%K|2Fk@R4i|8t0Q!d9+JV@?'
            'vIB-&U=oHix&@S5X_veLBZBtk<V>X|l>U1Tv?*K;D|Vn~cRaZ+*csJ#oFcuzP=Is}Lz@@68XIT*cITU7v6LwW_B$6xU0ntEQh(Sg'
            'BeFA@B^dIf8e>3N)5$%sxfH^bwCNe9xskCBJ$385voR_jQ1HPVXyejM7zs0=N(vY((RbzibTw}9bXFo<!>bw%U9aPabS0HrG+bUy'
            'U7dTi?nd3rh_NNE&V{i?*Qm=j+-c2h>?d#a17r=%>WFn9<n7@;y~jmh7&03SEX}G3!)}V5Nbg>M<%`3X6>}<B)K<>0BQZ-|JsMAt'
            'NOK0K>S0i*hwgzXh(C7Q*Vrz*ucbo+v@Fyo?q|=mi&~=DUoMVL>UL)#NHZgq@%TE8rXdmr`KH|ivX=eLh3!7opc`D<8Va+1TAH}_'
            '^l<IpY0n$CD_<O35_s2)Ln_-iv){IV%n|uWR=!T-5>u*GdAJ@v_C*q8(pc`kxAJ_|{=tDU7f|(hx;ww!&b)T*U%Fp$lY8YeYyU`F'
            'qO)BI&hCQNnEC9kjAuUqdi3J(8pG}|e`Or@;KV6;4DdhO$Z8;P*fF`+WJ6sCPux@4kJg)HLcGXeFvd95auQPFja4Ak78vNt%}`>1'
            'n`2t!D&tO-woI34G4hxBVu5TGRUp+YOBGcJL?&hzeuZAli{TNY#UcJe2j+f9SF#D|JF$vBAbZ4l&=OtmgWX6d#t)96GOvS#s+=`)'
            'BT2I_!w$JntDP5Ztegcn(-e|WoImY`%ooR%)!CFpi4fN1B*wT#j6K+JiKUKdZKBjpAU@p$o4l*aK4}46eqqR**St)aWqW?~26-~c'
            '5el}Dj;c!odEN-akFR937%+BPo`-`}Qj{sM(!_~S{%M)_jTIiqLDGooMt%a<!H_R$$mdd4dRc|x+Cy>J-6|$TayiQ&7dM!z1=&Jo'
            'OgAA6=PjT@$|YM;;|7HGSfV=TFpw&THjI_EFPBt*r9xsgQ~4vTvITc9eOV4i&WCsY^rxd9T!O%xYE0&J;3C5Cr>xU8ElNTh^k;^_'
            'AH8`2+#KQ+rQ~i{0-Z&$NwKodLc>9$BRMcNZ(r#9sCve(`s#TPYe5T2dn0F}Vofo>AIn1_mMSF<obh^!PM0%LYANGVFIP2~OYy*X'
            'UNf~_5_``Fp3;wnmbVO7TEY+{Dbw)k!2=Z8@-PI#0ZU0$&{$l7qldm4ShCwr9r%qnA#NLhrV~;@W-RtdkDl%W8LiRn0|Qfp6H=*B'
            'G_NXXhJ%1xRDx4q=KSb@+i95teqsqTpj6Lf^nQ}3aY$fD+`=<>g{rO82y0U6F5r&Bgl0zmHC$;5bM+V!4`FV;Ol75_d>ti;5^cUl'
            'cEo@9M04y^ie`4r6Dz@tT!JoYnFXy#ROM6kJJYAiC=J{UxKDIthDCNAs6^fsRlVK$&8;W2AjQRNyZhbRX!Vi(+5@ftw~V>v%gw{K'
            '?_Z_-mvysSAXbe$E3C2+3_72|J3`T>2932}+v>AoX2_h}cQQD5vozlX=ww3WRae0b6UH@t1IsqoGHFUpE~bw#=W6Y^-PS5l-sF){'
            'FH37=r!bOn*30K|cT~K<vENx8rU`Fl0!(+jeflKj#oTYt0%{}x<=mg2mG;w|#+PRQY@DN$3e@=C-zp<${|i!r8pD&BL8+|hD#|%!'
            '-8>5889HtQMMaOr(V#y#Fb+TfEYvP+qW)dKu?jQ4Gd=YJg;|KdlRBC;*2^?gxKHv9dz%F4B+fr4-LCkdn!%odV0xbH-8m<`nj=m$'
            ')|Gcl$&*^gSgWqS3u@qZ7gBG)BX1U5MU;7$O1_Ja0WvDHV6_OyU9gTyXd&B_`KpBbZm6>68bhc^%RgS;)PJTF`#Yr+`*W38NTrJS'
            'XGXDqW)%BpMzOD(QS2Vc#O}@`#&<vKkI_0rrelm8wDNdad}Gy^dB(4u?dvGPME)!Ea<&8f4I*;S<T4s=BX{mMPJCBZnSZpb4WFE}'
            ';U719!k0!5w?q#A4Wov?AY$0Y!@pUiFdr2g2l$zZ0CfsgGurruvBckDnyDLOx?dJ+?3;7=cMd%Mj)F_RDEzn>eEqOv^{P={VcwQs'
            'BR=^&;3xk;0m@dLgTGIpaxwU4fbxIh0A;!K4|qFz%6-fTT#h($S#<d0qXO03&oJKZ)tl3!H{#^Y(O+JP7sr2k_4>uDcV;x8M|wrf'
            'J{AZH76chB#Z-ntg6Z`!g~?YBalT_df5U)h*F%B~Oe5r}mq35|vW(?D`CMdSnF@8hvY7Xtdfje+8Gevnx83vM!IKMVP5DJx$84h*'
            'Cu@Cubawnso0t092}e7#AYJmRI=JYBHX%zhiaHYzeG8N$rNxF!HK~ZxWKo&cP(TdhIb6G$DzA$+;yvYm7Zg<@+^8T|!90{@w4_PR'
            'F^M_mu#ObjB*ZE?uKJwXm#(&wDeD*;@@h7+JA>0zt<=huXt(vGEh4@0i)jMYX4<$L8e-L0>dnU5tIgK`-UHJN6PcZs34KD_Mk)1<'
            '6_)deoAAFeVD$f=QTnF+WycsQ9h@7|ZxUnoy`;g|%nh)u2+}hmj=<ZkF*Eiogd@fl8kugcK@WIWT7f$T(`oY069tTF72|QiDCOFC'
            'WgZJA{;%FVKYo4u^YQ5`@#FEIU%&dx$vZb=VTXg8OS%GPz)4`4qBO|@o^G(oI5{F5XOP2v(dvRg8f2<D9cA67Y)0tA=%R(Kn{Qf%'
            '??9IE`T}$8!ic6AY@K77Yq+B}81SCvm}?g`)-~WDA$_I5{Dy)3y+W*;JpEJ%LBp%o?zT&y3&DQ-sZrW?#l62%ymqztb#t>fY%@gv'
            '<HdWohi!jKT=X45-Tz*J+`GcFzg|f8r-flV3JmlVc2pWWBMc3l26Ww_@Nm4F9qk7O12~U9DH!of0ub9{4%;HW?h_C7AIHu93q0pP'
            '$xZ)1rJMaUMxQs!IZXQpze#)PTKmsw-@Qco%W*Yg+(Zi2*$6<>v_$XBKDyy)e+3pPQA+Z~Rk-oB(j4SG^hqW!0@3gHbq`j_a}QR6'
            'VQ3EPeSGn)(#w1I0L3|>36g9}IPSFm{{Z&^<8%'
        ),
    },
    'video_analysis': {
        'sha256': 'bcee971d6885d7e58b40f37514f93ca63c0e10cb44904470d936a7608d6f45f0',
        'encoding': 'zlib+b85',
        'data': (
            'c-qZ<YjfK;w%~XD3XHmHEv6+!c6a*Db&q;0lQ=V}$)hVyy6>#lB~ub*aZHiAeArQPUA4b#zijRQ+VcVkfTXNsy0@mQW@3@V0dR0|'
            'aNg+m`|qNv2<J(pZsTQ?jlwicc12vM$7-FGqfNGn(o!v>GFp^zmZ~JXju)y}hk3MA^IdOpJfns)-EaoYxYan`?$Wsm)1{hBX1lB^'
            't9djFt1{b!2y3<pi!#djvvIHYCM%<PmffhyvAT((ZK2|_P}yCoHdRu_qXj^Umth&Hd6X{Jn=rqL(`)tNr5Y@<%{I*OR}tMtd6@Km'
            'UB|^X%GD-YMu{roPf;;c#Ue^0nM{_$0~}kn#leJ$diClxe39i*UO<B-bhD#>gJEx8m8x7vDqNHmd>JO!Sss_`4Gys$O`%33rw@P&'
            'eN@Xh8*QWTrnik3IArb@;U(Du!145o_!>uHVI2p9b6RCNd=et`B)*N*I?5weWW9M-u2m{nD-?iA2x{~Z#;J;M6R8DEU&SflIfR+6'
            ')FzD668?u^m(@0j7l6-UFHV=yHiG|4%ValN=JBe;FBaK0jtW?s_kc}AWd(GSB8C;T&B7%tPp-DrJc)~SRKT=$<r*6Ph-k^uWT)~7'
            'KMG}{L@B*h3{VgQcYs4^#qYLxbQ?!^YLUcSq&}QT9+g#|0-?p}XcKL+e5YXjLsgV{1uKz9<K8ovG%nb{;2}^zo6xk_W@!;4Wdr`G'
            'FPsvfcgE-~#{VdQpvz1x<KkxA>-YP;Ri16sY__Tp_-v-)%{I$R7%V_1nk{-gd6q}|ucDf7^K20ng?>`Tn}}g;!*ZR(a}DTc_{lGj'
            'j`1`7M*kjC<4;*?z^}ssLFk`2Xfuek+H7|!EL6Jf_0HcvegENn_RG^(AI{FzRDJF#_^%&*gnba>jKicqRQ)BaUy^O{!EBp^i)fuC'
            '%P7ZJyacm6`p+tc{oNljgm4S<yT#wC542RRR`DXn=C~O6a23Lm(rfr|SrOsSaOrmd;vPo!;pd;<zI%W6e0Ki!!@Fl^vtM35KYKen'
            '`;Yf$Z_Z!7ePfMwyxD$_fX18bmjAusfA6CChW-d|V}9}dEkl3t_8Cn3-PvCR(x6|IVP4`#n|QfQBK!-88~<5U1+brf(ChV<(F&;~'
            'OKzi@dF5ex9St4<tH`Q+5zUrW&b+FK7DFH;Xb%&+h9}cEuttoaj{lI7YXF;AnkXK_j0Raheh?CW5963lh8}Euc=G)uP@|tz2|F{o'
            'Si$ZrVZiv$<td{Uc|A=R_T#$Rgz0FN!!j;mn<iO$J)+&Icy_ogfZt|mjC&K{XXz3d0RlSv7#Fmc^p+#egPRb!Hg4QFmnNfmSitrT'
            '^XrPYoO%!2`WcW0@LvM<4E9t8dlWh|e5!y!sjDj++gDd=n^eGq;<N&u!Ek7?6mX_6xeIqik5LtFw$LSv-(uLjn(&Y6CvfPf<B5jU'
            '$DUy#&}no7+ZA4(y?rs(7^8pVm9z3NPBq}A?DvCHvv9OPK!1evfxBkNCntgVRIi--wY{F$0GHG>fkAKwe^L{CX+ZpGswSrn_2gk('
            'MCz9?siLzy&+<Xv8_ZA8L;)Q%d;JN4Cz~*d`*qjMhPnHp>9`d&h)F<?vnA`nH3qio&*LOcH_<Li9YCk*v;FfA`Mr#%%y$;X-A!t+'
            '*|K@<+5Be}0ii8t53B-ENT436A131oHI6<mqHU=L?{`}s{?M9AfLs{9*fZHaZ?FQ6mLf*<BiD&P=?6u)iwmGtjebSv1c20t_GjxZ'
            'e6zdiQ<d_OtFmQy9V_74sk#YwX?UZYvG5|qyDS$h$E!9pJ=CE)pJjLx--IxoFo*soM@!h@^J-V!xCCY`0wCO9x?#&Wf>z3#PL=Te'
            '=8iNgn6$N$L!#NtAB-BA(FP!)oa}{)2Tl`YH$(wdM0F#F)>}A?caAv3`V|D*zNpzJke`8#C%|k$^5<23kX(I^^|igE(KTRag;cZ&'
            'KUq9Uqk?FL8Pg{GIDqL7_aHf70K(AntV)-Ix*38AhUz=`AHS-p@H;Ir%xLVhcr}A_V-dl*w;X`ba}6Y4(D{o81J8D`jutl{Do@pQ'
            '1kz)fn-5XcYJdUp1lt<wmgY6)<iVgGN|nw;@|^L@<zTT{4wV^Deewe<2mwNzER{4-cWV^H*ICM11|F9SK8=q+naDB_8)5dqr-011'
            'Ky%@>2mn`Cpl@xel#~ZJnKwX)Y6&xf4UQ_wHr|3*|Lb30zB(hQx<eQ*oZLTWNkUppoEIg*LtOx+MSL2rKqb5j^JPII!4e*WesBYW'
            'g%je%3pm@aRkT6HFiNxPdJV#Rc%5b-kSq!~^a&m)7@(ukFvcLQpq2yr$cOhYMt_8UNaiK|Doj8iK)nM+6V}3TWGFDa{@X7EY6B}r'
            'DhHmp5o%MQQo>xqYnUNX{7uZ$3DvKlMP+vd^-J>s-PVyK5D&-$jJ`1b8IYU~>J`StE6OFP%2WHRmp`A;^C-_7&ueZGeJrQ%^9sZg'
            'P)IYx=CogxtI;2UvLf7O#k4QeZ#ztx91wuJSsPr4w;v8t4+<uQV;WmzF+ep2=P?<d1dZnN9Mxg_iSR0GcJCPgtRLzk%^`rSRyYs$'
            'Zq0@;Q+M<%+knUnC(O_M!0e&Lm_Np4A9_;`Y?0v4AmyIXpD4Q643MB~f#L@N3aT}8Ftfl9faHOXA=ZL!=HH3o?}FafVUZvr{SJct'
            'Jq7Ne?<r5hZb_mT*gs~=I5*swpP)WG1*sW1B`U)W)dcxPoNlX9B$T_jT*LSk>!iHV(14Jlbpv=i8Y6bdS%fz|g-3eFUtKl&25xl+'
            'Y6i2DqT0fj(Q>TbLbv%H9!l7M8r?<0nm_{@MVoE8qmM!52GxQG2F%Wb53>dEu*yZ0M2iyPzIUPJXg`4u&^m-Gj(XHW4K||q<Cyf^'
            '0yYw9%?f$p7N$>TDC-P^;o!8?JX&QrOO&X`HuRSUB2F!u5p+#yMFx#e9UTM(eBELU3Nwy(5VRfdFW}9atbBndsh0NtW(%M5Dyd+q'
            '!1qDg*D&w<PIsm&IGoEt0RPO;mUoEK<M);o9o+~=(65+|v<wmFt$2aw=Ab6+(R_|bfq!hA9Egt_P))PEa`93F#v!v@q3xgr4DF&<'
            '^FyEmG<DG*K%VxPfZk|O4sU|*>LoO9knHFc8G1gBlk5(K4utri9j66|yKbOg(l?hj^6Cj$0<+m}AWNgU=N}nq2l#04;5dTdLF7Bc'
            'v+CQ3CPir(xvEmVDr5B=6_Oj+sIXzHPhkpjLzv@Y=|SK34R~2JAu<#;>*)^FO|+YWXggnqgsxLH@UTX_VV1(D^a5FfUJ;mN?;CZF'
            'j4;|{c=U|!vV5u5QI#`i0SO=Z8OV3nG1`_>%fhaW%Xm<ZduOlTz6Zq3etrAyxn+$j)*yQ!`sok&9RHq2WghP8zu2sZ=ZOB7ZGj-W'
            'm23IK*b+gsBa2~ty(SACsOEqg{Hx3>{+tZV_(wR;=vaiA>E=mxOXj>apif&KLER+NBVvWRL$eV5HBYjPhmwca2n2dr=WqsVaPc}!'
            'x%VQYd1gtq;eSE)r|02qSl}Pm@u!3gj{tiS(Z{H+Xh>zWPU0IH$W2H+MgIUfVNLeYGQ`QEt&$E-5Y+N;QO?NR0K_mO7r|h)Lg{uk'
            'kJD4MaDx&O-dSvT$b>k9(@=g>v-<Zw3!*Z{pHn5aLEAcF9%^}sY?7Wn1INj;3Iyc~18Z1={njzIC_J*Wl{W)mA!VjQIaqVl8$ZDY'
            '|LgeVhwmRhefo^-QF0Cnj?i~-{KL&(KdBAqDeBj!zp&I@0s+wHEbs%@gU{1qk;f#1J!YqiIS$o3<N}2PH3;_;aR3r6pjnlojI->C'
            'Sc~E;$Q8U8O6Y5>k5YFD^wHoxW!~d1j{eep8nnP1Z^Im&XPX;j*!-i=ay?pJXW0$?Z0qI=8uyH08N^-ZAgSpPNOwdVeYA!#&gNm7'
            '^2Qn^*>!?`1R`|AX1^xCZ`4Z^;6~&_BeN=qJU~-0mKc~2E*9FG<$Quu8Afluzq%R~#nqJs>@KT7X=UY$7k@H2olI1<MI-M3!9^cK'
            '6yXCg>Z+6PwLm0p9}65ZL{dG~d{=^g_$%saBhr}Au?V~sMOLk90viQgQY77?_O4j~je&|hH?Sc#Th?M}?xa~~q6SBch?HF5`l}o{'
            'xdGFMS@&@}<pZ;&0z~u*^=@K7-(E-QwxmBtv3%?9uV84wE9DbsU)mpR!Y#s@oZg;J2&j1wo=*OI-~RM=MXV5n(P&B=W)R;!J_#P3'
            'oXW2?{c_qYPRVbR#Pg4^y{Q90K*8J=h4xigl>9jUI9qM#?_G8K{mEqO5YzMQF5Sc*5sT<9j<T_Os-BFGjZweKZ)08@q%&T~5M8(*'
            ')nbh#n-n%Vtuv4ZUOoN$+Yj$+^*%;7WKq_h*%TO%LY^c5LHZ#B$r`O3qmc#BI4-NQB<F69j?^5zwZzHtXjLKa0STKp&e4ao7H(vK'
            '7PurtPr2&<U<Thm!Wjd~N^-`WuhH{|UJtch19Jv0w;av$?5==Lfpp@POP^u7y9;+j)QmrDOLiKz_~byApjL$WbyS+^lk--FLMoO8'
            'a0v?nlof>=K~9bkud5I^a2c`hC0w8)YSVYPzy<bcgpS9M-;KiCEMCso<Og~s)A}a7iDsXoJYyQoZtD$!6eO-b%PyGv?09HNoOwik'
            ')2TV=L`j#w3zA3i$Rks%n!Tx#<Y#$3FcN=TgHsDbsP+kU-jPv=-CHFnhAZGPAO)s7?SLX)I9BIX4(k>P-#M}=b`ynYQeO(6BsQbi'
            'as?coO`OXt;-Mrc-T{~V1e6Cy<mB-W^&$}S94q6%MlzjMz~PGhRtG?r{qY#>j(kWA8Zw7{c<9sR;>qcyC0g(rG{u!NE5pQfYO<l*'
            'VPOqphjXTP+qCV^K=aZnS3Hju2z>=e=T%ZGql<grqV1aI1ug9+Sef=vsLduD+oYua(Be<|l3@}&7FjMv-%a3HyfjK8%LTvn(O#cC'
            '{vN|qB5fQaPE(frkAHxH@^+_Pe@uT1C)h4Fv*P#%a@k`qp15T1<b5k(CjXl)0Xnaci7(=tFpnQx0TT1IL`p$VDB$z3SazqlB+`jV'
            '*hek50D%~>zJ&8+Q9oX6tA3Ykk8w}HL}uk~TQ>kH8%@4E%UPVRvQyyqSpx4~gh0hzz+Bni$o%ikGzwMEfX}iOMR1MoIM@orZ0fTB'
            'sSNn$3@y^Ij?p#w{&@}VD0nFf7i;|jnds*uBlR2&)e$fo{U^e}f5Y4B8BivSael-TI7+h_^cpSJGk&U$Q2i~APSu42uLegSz(4qX'
            'h)&OPcXWSw?^?@f*Ym_?GBM3EC8+E*t{rbP4lkQJZwB?+>cyP$KSO81ruGlUj>gpt4K_oCYll8Agg36qY2ci8^_svgME_Y8UPlsY'
            'GKWK&1~PpD>SMS>TMs~Hjle*`7kH9v&@5R@KlkBif>sHf?v@jMQ}j>OlgZ?Ms1S`aBkdL*I1K!VsY!-g6K0-O1rfuP>CbVsSdD|<'
            '>WJ28MBaC4G(<yhsf*`&|F?M%YzAV(9*0|um>bA%aj2lw2wTwDtD@^MJ_8;Bn&}|W<FLm-=2`DMXBsn<?HW_DL0C>4Tp9O!-G*#K'
            '7`lR;(N009*2Se~yoJ%=%P~7x#b=?U{TL&2TKyQZ@E`$Kk@LcgNF(fXqg~%@R-iEUyz;?c4}{&a&6cvY+uFgau2%RSn$Ty`2cf;?'
            'Q;Su!bz;4y5TEe``uyGlbl;$!JqYPU1}<9{E;{glP1@1WM{0wweZup;vbb&wXzOjutots1ZfWm+&<29YR91>`R=>ZuB30^x&Nu2i'
            'M-W3$9=<ip?9%}{(7t*EH?-bTuotc|{A0E={A>pAnKrXF;%Q3EgpZDH9S}pJuXZ}S@9mwnh$I}AJfj%pVqn`Abx?FIcEDza*6gZF'
            ';-b80i7dRd0}KlF^X?oWKPOl!ybdv<Pu1Owt!|E_gCm^J5oqCCRQitiEY@;FNIjz^abOu8h1dfg1V=m;W9R&)8FFX(@TQ@?4pYD?'
            '9IfbhXVXb*qp=AR*Ug{G2%pP~A%-sE1oQ(I2z0yIB0(Pf+2HZi{~i@zfe9~t@b_b&h=GP?Xx#zdV8r|5>xytqk?G6m1|U=*zG`TH'
            '$nVu)E*_0jB|{!6J-$F|6|9a-E$j_yWq?EwIk^y%vkfB~zPE8dX!&FuE|)Vsnt?sZK^w$z3mw~^5z$P&Hh|Ar{dy4S^;)T!#J7@1'
            '9o2$hPYjl9G2XSh0RtB(O%=5zHlP7YGXM>eeIp1^_Py&!x)_-L^r2uLAj=m~gCnEp)J`qrTnazX5~#?KQFv20T1uI1XFw3>cb=-_'
            '$q<bevn2pP@yK!<;M?(&JsyZP00Qc|Mi&=Ddvbh2tYsN5O2b87QBHxSWq|lqkZqT&O5_O$BwmO(1U3A8M6xId{^@p{F2g)Wd)HCE'
            'X*{ACvVuT<HP#C7Ly&66<MHLaIDg4FH{U7t%)_41xe`mXyrh{b?=nTuDBA#)AwaN?9mA7^hkzQNXGO>hi6%MR8}e|B`4oeOKwsoj'
            '(wIdy-V|1gRTNSZ1Uwz=rU0+T^0fXlxR|gYNJz4qqr?w_c}AlA01y>&_&c;40L*~9FY+$HJP*VI?%gm>JSjC^enhe(z}JXVf_ia!'
            'Iy%0zOexr&Y+6%mR}~k~`qJ@NgLXU!=f$9IFj64DZ!Y?JE&G=N#y0!Qz=MG=aWwXDJ<->GWrVXDxIV#6{~RjV&q*_~g&TOn!yl?2'
            'f_rBYh+Ag}7%a-$tk)hP<&283XgQ5>R;NuY*qnlAfJ3Bz>j7`@Z=zxNW%r`}lo(Ws4(nSyP&Bgm2dot?Ye5@JK7f5fYAxY_hd|zx'
            'b$q=xYs;UGF>Dj*%Y6ua1?H%uW+Mcl{-lnh(eX(`x`QF0p@9fi28q1#i8FtB!=Ny3cThtPj7N5heppTnn$2-3!#s|PLDOw2!iWC7'
            '*XgRV+bB<fl0em+8D)*7Ddc;X4>V_nT@Oc^vg|c9(LP2;KWtEdW;qB!OpMaP^#Q|5{V<+9G%yj{;n?P>S*<7I@18u+d<9#19xiUW'
            '0Q}pNj#QQ*9T&CKc}D97Vj|`3Ew!~>2aQpt8i?wf4QI2Zq@R<l7nc$+c1}}ahq5de`}qVG2w8EuNGdo3*l);DU928&!W}z#Hqey7'
            'k>dpChB2(<2~lYj=9o%IiGi}1z^pmml01qq1%!k^2!Q6G2xF{%#STico00WH1FS*8{Lyx3@Tvf>_-$20OBSdpfzO1xl7}Z*5}>Fk'
            'M;elvN9z#HH8~?mTw44+utV)h1$MefvV!t0*IB8({b8yy6G(TRWlPMqQ<4no*x2l#e;t0!te`>D!Vq)}%gh9R>vqUWM5nYor(Mkv'
            '*WB2JXv5L^g#)E$(LJ?rqp!Sa$Y#H=w0St2M*3}I`&crgXR2yrAx?-rKO=R)>MgJ>WtQ1ox`ruw<A>ST;qKlfv~>Zd@a^wC1>INI'
            '1Y{S_>;!-de=Q$53nk~FZdsGSg-sT9D@4wvta#0QDSLy$&py*a*VuYAXp%uqO?q=V<U}Yk+s?@5b#H7C71_ZUQJv9TnD#y}>oY*F'
            'dGn{Ki#;u=C2$hr+Wd3Sf127DzHA-%P7vQRp@7Z2&54qgSEsc!2D=jyXq!dANkT2tBbX-h#$;SLq%~?8{hzcL({YAc0%e%Q*J-pg'
            'blL#c^Fp&q7)^AI^XZ#!9je&%TO5&oH--TH$?&+_PTnBcGZeo;zgK5pHI`!J?6I#v7H`ca<XWLd?qCfsr70eaCE5B8ud|2z?O|y0'
            'N6~?#Z`AW>h4a0|_>o&6y^!QT8t-FfHas-qlqdp__`uRAow_X72sD;R#_DCsDL0&nPf|l^L=q7j@Ql{7g%(Hn3~o>iV8eMf^bXsg'
            '*MdSIGwaFFIbZ`fb!KW=T>IM$^k}q(_IdszLH}qO=Ug!%Oh$yD(e1JAK6c0CC}ckOw~gI>dtmLYX#7qaIR~6A<g|6(>~HLMYd!ZH'
            'I-8Ho(e?xDOPl`SJNqYpb>B>L^Om?ya&wjuAM1q^-H1qVX4Op?)S;H`N{v=DNyFT)!*uBhovMwO3vV7fma1<3gSU{YZ+s`&e>nz1'
            'xHhfH9!=<<Qlq|FfqGbV+B;wo#b$(5yWPp2c5yl*#bE}sMXz|f0r0yg<4Nr-aP_oj7>jvWu@h5UMGa!CM?x`g96`x$e*!ZiFbrjM'
            'dvb(Cy9R~Bj&Qf?vS?R?UF${c#Z>Pl7^`R5W*(;?8wg)s;*yp-A>a|HLiHoE(80Ega=qc40DSXZy;1O#S>b#z2mqOT5`CnZE#Yv;'
            '@oPIc`Yyuocu+VeqvOeh2dOX)p-5s1AZ0R^Q0ePDik2~)8=QAkaAYi-{BT!DAJ8#oGg-p8o||w0fvU?qG<`63L7*5r8o>N+hVF@i'
            'MDbXd-q22^X};x&VWdh};7MGgivuM<@K5n{hmT-|2-b6qb9?jly}Cy1V=W-JZVVRSi6ZlMCT}U!z8D0q7D|*flUdiMOU)y+l2;o>'
            'S+d8Y&u32{`qZpN{B+;qKa*pdwHV_TTCALY$PQ?QBDD?zZEY6-R4XUO#7|aE5Y3%dr>?vbEwrfYD$6j=dgwumRjHPH+7vlQixd~P'
            '9iz~4SorY<@Z8cd)lb%(aI4U3OOIB6(i1X(E}bk`F9PF&o!yuAs9I;Dhr*v5FRv3^QBW}Ccm)_kdvxBvcslwY;po%<`Cp^|`@fGa'
            'zwHlc2qs6zjm$uAZWCkp>cWXMxOB6V_K&;qt&a3SA3N#P>_#7Aut#gG2YqDs-&DR`EVGMZ6BajECV?Fbp#Rkd81>)uW3yO+PuuLw'
            'KwEhY?y^xLMH4RUI!(qSWjTY7$5xRPXGec&e1&B_IxxPXHWW4ED-7AA;T`?)?)$cdgfH73x5Pw%bXk#4nfM5<uHU~17;+RjU#iCr'
            'a{Y!R($wvWYGd(=KcP*qUQ;4Be9~gUc%UC%b3Q}+B1NDNYK!Srm{AuEYC0LJf1d<_$2ifMWUK@JdhvPe%|mYtH`~qLG6q9PZx(XZ'
            'arXkx>a*RR$R-N8Iq!NNzQqMPMlOQE%m@YmlSWcRrHrP1d|3Vi*}JpQCv;@t9&sXPdOCQNkJy`N=W-=?b%s7w3>)-iD<>Z6hh4ZD'
            'woTR!@wIZqK!=1uo*I|b#oYz`W|n@33+L?oOSZ4U7x>DZ=e;Fi6|QdTktE3BlVNj1kgVLY!6xJ5CwtH!A9NEAyq*l(*B=(~iLKa?'
            'KQmoMABV<Ek;0a&HYD^699&pd7GI_^Shaatjda5B1@&`@#08@0P@O!8+M2^(^!*;;5S})M(P%}F=~EsUZCze4UiC)}7&SG>$mII~'
            'M_fmGIO-4Vlod#tIeutiDMlTqkaCxvp8;D~qKY(j4CEu(nk(1%Z$7mh0+!(olX{{;+he?b&{^3D(Eh5W58r3%Opp19?tF-D)_=Fq'
            'C5)EUz5#lfgN6@wz|Ln|KKoW2(iT>@T2hZgL5wBA4Hu~Eds6Z`nc5vvcG9J7&$kc^Y7(UBaEha~q=jrJj4?p5t=HnD6{0Ys038HV'
            '#nJ$dyV{QFeC$e7<cXXY0~c!sUQp`@`W3@(mPh%?SWfq;-a$j~V}OW03&~a{x+VA3C~@NsJTsAF9MTq6ZQBB5b~?_R!ND329u-^9'
            '>wVSTdc4iHgLdhO<lCmfsR^v1;S9TsIn$2kO_)uC&b0Zc*{sc+Gi`h5Hg%WYLLn;T#5qw4u!sP)(c`X04gKBXz2cR%zaVk<Ay_q6'
            'YKNr5lDpveMgt3wSIqER)i7;70JJ%_PLN_0?E>iU2m<LpbQ*u7p4vHqb2N)pOD=~CLR*444x<s;_hV3Q=3!YTXfaY;!C<VOqKx7}'
            'WQhgf<>cH#xdqKN#(%)MpA|YAbl*rPg~Eyjm>=rFBPYJaj;TsASM$A%!lcldjGW~h>V#J~lc9GcTn`?l3Z*!Nc^Q`&tE<jS;`3M~'
            'nA1>euFDxh_>hW}da$^lCyWJn9Wc9M+_g08u#o6-S(GfPd2y{4izXFx?<g`(5sa4DZI&6QpX&xkzG!lEd2wV*mPeQOAm+7zB0lpV'
            'C`TOahD9@f7?>l(c0v<b_Cau-%nR_E(7siJexJOp9cx8;QD6O+`rF$#AI@JtReyi`HwAp7+=ZWFRe{(6y5T1J6!e2VJkhT6EOvnU'
            'B=!UH97g$2Jf6rs_6f_5k`AL|qpXJ=c5gv(T__p`meAT4Oywl)vFByBilby{0{MlN^*gG0gp}*1{_SM4b!@3*Z?5Ba*gS2I3V0OO'
            '#FmgC%ILF<!xU<XrRF5*UAM$(AE^Px{Od=QWO@{s;)3-YuC9zt`)(a$loq^7;uOn7I5oZELLqa+z${pauu4j)TrG2#dh%Q>EYkIh'
            'xww|9Qp~i!jsRuZY6WU`Ar-{KCFT=PxQH>wGh_v2wqZ;dm8(EDMFD15<ye=_NA!)=iz-P*k_B#Ft+7@&^KT5FN+P!vacba-Sldp@'
            'P2}O7RSHeX8P?aC`s|yX7g>UZFHA+UM|h3_V(bZFG?=cj+$a!MBVE_<9T@BlO|QYBxdt{Jey)Fbi3ZYvt`VnqG#|tu>vOavYIIX9'
            '(Nx2^w*-Ts|4JY4zg0G&MfAFii}k4S&7qZBDBZn9QnQ9Vro(A48S=+NAl0~>_HTev`a@lZa=PkI9)G3}?qL_ipO8BWCQ&I^5@RsC'
            '*s>E?y-FpY>2YA^xYt`&2e1el9s|zn*0az2@5K>{6-Sq+lau8=6>_iRqWFWp4U^7P$G~j5u)+hmq0j;9xbnmh$)#OBvte8RXEWZt'
            ';w+8MgJB~n-Ydo8mSgd!=zaK&dL3>}DY|4gR=<kygn9KtU6uwfl3@10u1G1zfgHjEhu^$@d;Z>36CoB8>hcsQGwzTVQbCYVR|w&t'
            '+9%^<A6kylrCgEcI+nN-eQGS@)hf!Vuqq9zgdS`51|U6PA~phB8mpuKek44a@{V;uy#j?|tf4dZ(0Oq2nq~o;(a9+sc2ad<PmLNl'
            'MDmb})OZ6wXC$p8#D*@~U@=iwTyvQEDF((?=}n4Pnfx&FOTIW|U$EK<gQ#oI<Wz-fC`TY8pRt5ua8~#gTHLZUa!!D%L3MPsx1Ekv'
            'pg$k6%`z>mrCPe`usAgCZG_9is%Q9B8#*dCOdAy-J9$iFOtoV)@$DZz5atJ$y27ol^>9qJ9(uM`2jX9aq6zVA)W23(;B$>deza?v'
            'E9*oe7{%7^V5KK?%1Ys-DC`UYF3%^3rS3Q<1oQa)hs0<6=)Ucw==Huib;OkdRnxM<cyBB{q^~_-!bI>Spr{k8#6_V+7m#oe90-e~'
            '+X8EmLd|%iL`Ij-pc=ZyOCN+7>1GEk>C7SWqBe7!1lcALpoW5R`leg12MZf&xvCP;c{g~J?F!g5Xb_7+lw*+<(WvZ(93EH3ITi#<'
            'Hsjul53gR$&YwMfBR3b!o_~1vlpICpXU{Mk=#LnN3SiC=Kbwq%#A^}<iPy0rOETqMNF}AUWY2f~R7<q*3O04c9)!pDR~Yo**K_<-'
            '@ROyuCK4`)I3YA~@C%D}#9LY>@cP56_b-2b_44c;4FAM>|N7~FSU08|JFnipK6~^2eD?F%JNX2IdvKV)l1*Kk7Os<Q&YCz~zC?FO'
            'i6ug!d#K1R$QsuyY@4d3xP}Ptt}X?_=^M3B4KfCDOUBh$okt+s&Ja{2HTJdOa%Io@FR$MI`RS|K*_*R>fBF0D*Rz*@`RjX%Zu;J<'
            'yh!Cq4y?N9RatzfMB*8DK`8q5uWw(S$x(JBd5w5PSQQO_e)gLD`p6YvPl*!rGj6YA%BfH*e`BUmv@EC`S`_(2%$tTg;{94@rO2w#'
            'o^UoL<$#hI92CAPFR)+2Kx^GZiIK5uC+fP1cao{1r3=<}lw2@oiz=@Ead@r7uP%4kCLjKG)sXx;eE;98A^j$5&KDfYUsp;=3hEYk'
            'J;;|-QbMwy*)}vBAeAk)la(o-)lW*EG?R><Wg&Wk-n~B8{0fhVE7SZ+g^N5ZP-N9x>({5h4A~$Miy)<U>hlPsM^dL?hb?t<8pyOd'
            '&GuAbVlN+5<y|(y9VQHbrRflD7rae**c9hQf?c83Rl3Y^yW`z$(RGM5E9Kt0t?&rk+RPF{+3V4WbEZcyuMsDO!$j!TuLx^2@o-ok'
            '#5^+kgQah|WesFIsaOtuid8z(Rm2si|9U0_&Tzg9Z=*UxfMe-3UV{fOkTIWjUl&6`MNSNH?L?&2j5}Ww&%yzY>DGlts&zk~_vuy?'
            'W>NOV8>~99D2d$bIx18B)D94fm&Cs6uW%vgsZ&G0jGT{X+PPb3A7e!b8f>eR2P>UxlL`>lXbwUcoGL@9Se>fR4Cs;i<2}A~3#`*%'
            '4!J_=I$B^n!Y7sGDfR|B&pQ5OUlT=iG~=vtahxAx4XbiMojhvLgJpEbi$|&r1$s;#ypaM;Vg`S>E4j~sJ%C1_8!|v>czVEH6uf&R'
            '>Jk$n=M`$U$mMm8GrWL_9sdxFLHB_N81O^W`|bpt-QS)3+Yf)VZsIXlx(Fftq>h^dkz*3?9pV9usbu=&$EF=dVKhpbHb*UgU7Rw+'
            'OTbaF203^zIv&=rI6x_cV{&@jz?L(DrmEGBH`<%p+HSHPcwHaM>LV$T<In9<Z(ZHmL{Sz0nb*-&^t*y%USlW-0^f6^JN$O)Ov1^C'
            'ua9A!8gC`DOY(x9%Wj?9oI7uQu1DV4!qB@34y$*7^_d+i=Y<(^J%rIQMpjsD0g2jj>qnE-8T^1YIWq@xW}eb%I8+a%w$~K^Ujq<8'
            'ES;KiVv?|k<{fQ$mZO2<q=&jL5(|W4`Y^>X>g*7fS8!15kmJ<|hVD~Dd0^YA3%e{Cg0X(0HI=^c1FB{1jlHoEyy{1RIx(znyr*;Z'
            '19=t^+Bi&igCOYc(y8~NX&U2SDe^Ti(hk{XW=3HWji>Mzu>F?G!)gGC64lZWt3>Y8gAW5tW(l2+^L<xZ$j>nd9(>U&tI=XsxWl5`'
            'wQXT&bZ1JyZ{#tEDE0!hQT9KhTBAW0Taw!_iUtHFk?gn#c(O4Xi^;8LmRXAvJ57tgmDw_hRc5KWJV{03xnSRu$-f6!@6Py&3Z<`h'
            '<BMuo`3E86%9!A0oD+97l!QKG+uLBU&kfy-h;DY)-(0cl3vNKvb;G{)I>9pB=o>LgEEb8^v(b%$`>ztT3Br>KtT%!4v3Dn+oE%M('
            '4H`SZDhlU9dT4g@EV+*KBwD4<f0B2hk6S3pa2;DOqoPKe<}-_-{iG(nwp6*L6+K;UxpWVT7<Dk8u&R0t)<r8xMuB*UXnRG5_EPz@'
            '0y2q*U@A*@Y)3S1!?fkX!^hlV&*c{_<ti!e6xA(Gy`fe?`*nyobpHGib-VW}K!4mpugSjl)vBb9`lm9!Z`JYekb0JFmkc)BwaNh2'
            '!qo9<&3WwwOkx0wF71~M_kC`J8M;*(<JIFc&DQ#zWi554DPvD+Wl+|WY<Yh>Qaq@YywOO)@~`e>T|nO_=}q6sbh0mng!Op{b?$Xj'
            '&|FX{g`^%tJ<8;h@src0IEx#-O#mBOW!V~nx3p|6EbqoiTP3P&dK1`I#tOx-*(AVy+&5<a)3h70O^II+Je7rZ7>$h3PiLd`b%)!T'
            'VQWp`pk?TQVoke!EofKzOyT}gQY0v#){e}1IAr;uU}ZC+?s#QWhiHn?d4t}=akQ`}&?g)%?Rapq63J8d+qR(FMl>-ur7(X&Lwg{x'
            '9_wjuZz*5~@o*@Xj=7#rT~V@y5Umy3TNkYn4X;Zm8kk;&*aIxSGlQ(zT}xuv9zD;Cnm6U`9)M_(coSMpZB&fTpGgbXcT<170<&KB'
            'i7u*F$CSwWiD}wsp-sol!2Rh*zd4#to!sB0NxTcw8{fs$^e?@9bN=%AS<}BHk!Ouc2Uo~?MsR-BsYvXoG{KK`YB~9KQIl}aGaZJD'
            'Ap$1AD^&k}!UeWM^`BK78J{4BUFetEd4r0|QPK1t5~s!Wt-HV~l-HEYz76%Iz8l4*HKeSV7eBBHZ_yx4u?3`k2>3PaR4%)TH<n<S'
            '0i}+l=<}0GfO)Jhxug|lMT+iS0v!q~OcC<UvcMz+I%l-A6Un%!v`dp0@8x-IT(Gn@b{v!LHX6xV*oFzjTu)yQtB&y_E*H9><XsAL'
            'q5DM(9eS)IA~FfLOX<!g@*x|SQ;AAp9g8EDI!{5Y6<L{|RA5tKBEmeA89Pq3Cxbq{5}!3m{jiVL;<E<S<6z<xb3h_5YEN69+#{(D'
            '^zt@XuglifelF@jh-M&u<6WKS!~32iU0_sHd67oNe-WpUK?A>+hqb7qOO%nu4q6gQjs%R-sz3_hA7u98PMqRUSSoUAeb@4tho+&C'
            'w^H>f-g-P~Xhs{H_F82$ZFQ0%gOceLs<cB&TGqW80@u>OPluD#AiW0smJ{8&<67)q5w$W0QxsEXKv{mC;rgNRyBX2-OLs^IOspZn'
            '4M0DDA?|nZ2gbADqogJF{DD}3N9v(z=stu}zF=*n4&gG&$;~0sqEB%hGK`BcPjD1{OKu%LP+FCde7VrsOw6fePoE()^5eV@JEhXS'
            'fL!5J&-<SePtBFh4+^OIOA=AhHbTNl&?%teV7W|6TL`X())s$jNKNN-;UBgDYi2I@apSdI7}z4n)4^pAhq&bIrsuH!M-SD0{aTx|'
            'T&6Q?x%w!<+vi5N-;H2i^Q?a$ptg&W%*#DjBOOw7xAQ)PgG){Q&W3i(OcZ+ykLADkG@5o;c)Y9;6U!}$Bh9ELCuZUW$g@^9NMVIY'
            'oV|Jf^4*zzrRR|9zF>iLN@LLCHPR<jou<qyy1A38E@cw<7hhLdRp>Oe!O8cN0%H^;tAh-_+M;g7Q;<9u(@oM+LJvh*n9~+7d(>mr'
            'Qngw|cjo#%Eq_Yjo-OxoGsG)^4YBC2A3A7BqLNf0vh|z+l0q=2I*#zwM`C{>>x3)d3QLqBPxZ}FAkWiX$9Tz>2_8nFTZ@epyo@P('
            'ShN>rNW!xwD_zSt(_$}1%oK9LBRWA#?eB>7y^j<HCIa_BWa!W&7fNj_S{4@5sQeVgk_^T5jyzEF^ulrD+`-v~(d1YpSQH|>mvGE+'
            'Z70ZYxWU<fygYyZ^37im`bgG%8*e$s2Sj&DM3SIAwE7SzjWLerz``B%RaBDxEB9n@YOh^zOjj8L*g{1m-3O{ij>H4{Q>{pboQueD'
            'jCcnlD-E9C)A$B(;6tNJ5Sxy$xN{n>P<I0bPHs|jXcDjJWFJ)OXnl5#=#N{eB#9C6qApOTDPL4@W$}Y6uY#9R5ACG5e_*&e?l_88'
            'q?9f6O0{fK#ySP*!uGl{+0%eaWUYPc%;{fAG?7e{YL$^Hj$t(mj6Cp^nBO39%{skw=%Dqe@y(&pg8!nfwaA1*nTv;Kmf^$ku(4?3'
            'xYp*?67P9s(Z!+}oC0c2W|Xb5!t7Z&8MgQRNbLzg4s;FeZ(#F%?SI1F>=da2qvHocgbTiU{e5NdNwJ2A)YS4xK6KAp9UiO$Qf(T0'
            '?2!EUF1+pQ07|k0Osy*ytUwVePSA`W!N>k#?Zk{6qk-Bt8v?c6>tlh!HQ_AT*t(>1Z_VDY=cYL5O}}w<YhWjG$dtQ7G(BkCST4pH'
            '*a4zLYflW-+jIIOXk6)vxfbSfUAD)d>gt3TWBdTHpeVnk^VA%vKjLZrkE*cmM@FuMVH}aVu43V)Xlnzh1TT)4?SVcp6I+161i(8%'
            '#=y)BJF|%st_;5~kV_X?oweaD7iKPE^cYMq7|&)YuKMg=-$3i<dSG$O!KzQ0q2WAClUUe=+J##QrNt^8$#sOH3{XBW_V79ebc74o'
            '3xm&g_xC#8aut(J&%eN!;?6nL6XRcz7XwcZ%f<TC2}^zbXExZu=+OFTT@USMvn&lZB_W1#&n#msE#A2M<|U7i#uXqSy%}uc0#j*S'
            '=55iK5R!M*3-FO%&vig|#@+bj-Wqo^^_~rTdsKjF%d?^5@R&OMx;-hB?&;w%20HDBR=-Wfu@ZsocK3M6*G@_JL$rvKr3rTU=Q;I>'
            '6V*ZIv&Ttg9DVP@gG2hWWS`Z;s<jo~hOiAdz>=;Z^*_*8?zvpF@!GzT&;C@#&TCXOhvVhr>YOUu7$CT4vjrS-GpTzZ%6K&A>(Y8x'
            '{Xf&v>+CR>He&26qNB5=FoXaYe6FK;F06}pFELEyhe>ch1}=0H--LF>2CBHC57(MLFtJ#Ee%27l*VLL->>kly0L7fytm~9?jQ}o|'
            'Oe5&#Li#MF31ID`htE?p*P%7y2Q+k>b@+9^<e?)HT<$rgbzRi@E@r#d?o4hOZd_O0teez&fAzF!*D==~yOx@kvQGAO$ZeJz=NfBa'
            '1AFI*A)?lOrp@4q4witR8{fvORtC{|0d6-(@zBk|Vdiep;7~{&^Vtj4xjNMC;Sg?OOksF5y7w(w^sLQfQ`c(N0>-;t1mjJR8_wL@'
            '35yH79PW~oDDIrpz6;NiOoS+OyZ;Wupc$dA!}n?^7b34;4xNYOHt+4N9M%${=x@y6pq;L~P;XQ0RAT?M2dmAYLc&X<N`Es!px*p@'
            '>hrf0fNoBpl`oODIo+2?+N@Yt+r%4gO5Y4_hQ*zeMf`Jn(2ow_(8J{49k&g*;T$a!Yj5SpakMk#M%#PO4<X{qn99p72Tl<X=YSel'
            'TNFw$+KX1qt^mwi1=Y)1c(vbasQW0<sY|*J^FrD>*OCwULTu0;B$DvHA#Q(-O2@+H+Z2<_jV>n=65n)^VSq)R-Kp@7uUxY;uS>Ft'
            'Y0J_<SevxsYBFp^!`|EQEQIXW=I;GSQ(LEF`P&y(i~tsGBFd7aIb9q{mq(N$X||KE#+lOLthG^1pH@+%^_1MTA8!KeLFMZZCCpLA'
            'JZ(wjXzh=qi3j_NAAFHA<S9h0vM(=*EI_RkNrYpqc<f7g^-4xJ&3wRg)jD3x<J`Vo0adbC^7l}w_hfGrx!cA{X0Bx%{3iMv*}Gc|'
            'ZDK)3Guu-bW)h0`1*@MWNVlqp^Dy}$dz)Jlwmn?TAbeP>;cZq(b4{{6Xz_OyTrA43xKtOzP&Y|D*G1;OU;+=8erqZPcJZa&VXiu~'
            '(GEQQ*{Z^1CK()7F^(r-*NmH*I$u4>krF%B6o@fSmbQE`(=Tfxcj`&md{i$E>@}<vgT5Y?^+%tZ%EP_2Ve5-2rv5EGQ@Wf-*Lj{k'
            'KYQ`?!>jia$Z>92UmUzo8`T}ORAD&d)b*dbr9r2Bw;Dck?Kxbi+A{$;v6{ABiO6>9g`FB|CUK%k*<S1>tmCp;(APde`(K*aPOFy0'
            '2%V?MdEI-qxNM#;0Xo5|duPd46qsd$M&C7S+}Ep!_U-0-?Xzr|8V3gNL(V<_88{S-fv|W%5J`=lXz-f2p02v7c6NE6+p!nqiEt&<'
            'ev8rYFr4<;Jv+l;LJR*s3-WB%k-H$(&yIb2k0qtpHel}~b~|Wk2(3DFT{W{?qa+1#Rbw;xsp+qqG}c8(y<z_Juh4GI_HwY|SV)Zs'
            'F25V!6*KWa3Q-wZq4^=Y6)e({%Oj4RYs6hKg(_;p8^(q&SOzVf)Z-Np_Hvu9XLAat<r7f`ZVo~RDu=Dk+P6A;WJVN}>s_m-8=@SF'
            'gBtj^fp}`=o=9j!o0PKQsgWaG#Q1&-T=uTBG-XN%NnnXi#8NL%tmSAmu3ABX#a67XFB`eB#jRzE+>l|46sO8YKB&+8!v*hubmEDr'
            '5EBZ!FetkmDrh@>pVQaDgA_v15?aa8##j?B+!Jry3b-r$xUC!7vOp^18j?*{CbFW7Ky;z(M|@oq0MsQ%sbH+NP87>Q+L}b;Tfx`f'
            'GzM7N$kX?OAGtAsseMs8tOlfq2rCguO=@X7ZqS`o9+gIG0y&&vyHWzj<s(ty0HkbT-#;>!rsN)0G;E;r-a9SY?2vR~w7uwVYix*K'
            'Q~mtmmX4jU1B(5Ptp&A>)e2xV-2w@f6|SVz(*71s(jV2YPv5<H`Q|UD_C0wRC=zY9<xUN-Vz|DGuVjwbx5FPb_+GEKNYLBl1c#W='
            'wKv%)+p26%-r(E>=a`!UG4M;_G8Yt81FB=0Rf~1Bq=E(JE)3Qru<khp>DHC?`c`FSH6D*m3)W7#g_dZRrb^!<t)%u307X^;h9uuo'
            'Tw=%9Ey@bE^}d$eoMIiGJo#Zpu6i;C;8y^f@wbexS^i&##Ts;@F@HV?>WVs4OoxT(&I&qwIYpJ;;L#c5-4`6=d}>#585~bezNZNE'
            'lkefIpZEKoVS)#vO*#b6N$8UI>*yo@HP96mY_&z=IAz|pY&pYAPdVktMFP;lIc0zl78Iy+QqPGB)SIf;(IUqCAuvFI8Lou!0X0@h'
            'Ij++-uv`{~fMW>Ocv)?>MXL@)<3aN{#{bO1ViCtv@>~unc$}`>VV8&%#*9Z?ndQZF&>y0Xeu^pS5<NM^;9nMw$2^KbzbaRwKlU9o'
            'dP^<i4O;NZN};HqCYNl`YnzDVayg;ux+w}FBySF79LPrD^?b(x%kuOejYfr~tcjjX=ah-s?GH_;RGb_2{FI7zh}XyKGc38W7f#z!'
            'Mhs|H0!Wz<MjD2?@=-$3c2{tQl<_KDloFqZQ5O0Uo?&E!r-Z9agclUzEW=PyYvlTZ=;$Lfc93e0A*vOI^*gsoNY}P!2_}nC{mmrA'
            'V-lHzm9)i?FdSRE1&)6kaphV#jZ?~UL1P%qQd>KgLi%1w3gDfakPD}xIp-Ec1QLdAir|kLyRVL*vw`d_I^KwBEFy}Z)Ag!AUvhxm'
            '8)B!$%B4Q6MRPb+;62XukGg=AG};%;$8L|=yR%<jp8b0EeD>$R1L_NvVSLV~AL|0Vi+B@b<S?N41~x5d#IZIvIo4ORS&-l*aIb9x'
            'xq(&hciW~3zG-=H4*&e@)!DQ6Fm64Mr*HnIw%Ml`=v1Md60Mj8+9!2o431$1uszEq1$fj0rtknanISmUcyOISK0RF6%iF)aH{{{q'
            'L<-h>J@1*MC+4?JdHUJu<KD8y{@BnHHqk9!f`}g`VU85X$vxhP0xP5|aClms55%z)__4_xA;Wzu_d`m(xD3RzAa#~z7SIc7Zv!$f'
            'srFkZ@-8)|eV~V4000915^p8m-~njx;zPTy3p=?>YK?Rsu~NWNWRBo)m&V8$L#KP*d3`TU%G8!ObG~qfaM^3Afp5};<i=F1(6oCG'
            '0H2+9e2{qqs`HFkv;CF~Dpps>-kqBUNHiORHbjD2OO<{HZc9@X!3X%y%#=g(I<_DjP!MqCBj1#P&x#!~U|@Pw*Rvjk?X7Ac$`wQ#'
            ')w-jEiu$<6_`y5e8!E+Cqs2dskYw}v$TXYW8~Pvedw<*aUxc63F^PHcr#Out8}$#p2OD^zU?br5j;Lbbl_R;m{W}WM$yr5rKZ2@s'
            'Y8M~qYhi&xW6;Rs%Ne$1hjlKp3-V#UY`)bR*{RaREGH7WlIV`(wDy3h+sHn>Fi_A2*m5MH<;>nwLodX%z8oMy1z@M%zI%T5?%Yb`'
            'XA=@~gkzMDLQ3S`KnF`Q3tI5*))A#m1FxeLn{-DR<r09a-cbTLB~zC=DO@*c-Zk|NNMWEF@~2#rzfV~dy8`WhMRk2m253yUFZy+g'
            'xXA>{Ru-&y!`C?swXwn^WKdZ~^)pD^k(!|0pfI-zphLvy6c(_-l|ca^dl_A+$Lea8C3xjD;MqaYRWP<qk9I?Qm5zrkU$Nm8SWdXS'
            'jp2Agv!bhR^gDX>*By=a$KABjv}x}r9@~=Rl)iDebK`@c9YS4arnS9(?YC}Tn&$&(e$fS^YY6`P#tbMOxG`{g>VIo?y8E#_-g|wu'
            '+roO)a5UKm;%W2Vg~l;QK7wYxV&6R-0F?Wn>uYm>*)7b&E>}d0U`5t>e(ro~bOic<b%C}I!m|DVe<4>weun1MF-SlZ)%6MHJ+8_H'
            '9_twv+QI|60nHd*Yb}Kzd1{KTxoH9DHhTcLFR+OO9Wdn#2?1#|E)Oa;6^i7e$c7^>G`0#1I+}?y8mIR8-WdgQ>$=y^?ew4*puxAi'
            '+Bb9!bH3^NFs#Z9)+Sz9=j<uRkT*z57hW|=Z@Y2C{Isni`z$8=4H$J*93Td(D30X+@gB$1bBkgH9b9Gvcpga<4+WklT)q{w{%{r5'
            'Mk}q&UtHSyFdR-PUr0MJ0EcWrYm`B~Xh<t6jfxcWvx@9fQeo2Ib)57s?FlcPYZra#Xy}VXLnHPKJ0y}lr_j*d;X`*nAGWWC9&Osb'
            'Xgt~ZhBeY|cz#Y{5$FE3nnv8PHd+Z&C%K7K6mCIuIQq=jp&hB{rigE9MXO;`;pO4St~Q32D>+geya!m7+^S&hj;T2$w>f5Jl~l0Y'
            't`pkq)LmNYvnlX)4#H`z8be}Q>*`uFEb7yU&ks!4+Ed(KThW$&Z@9w9&b?l3>I*#f-r2WpxB`_+R~xm6(CDqEBZowVz3Q>q?_4$4'
            'uv6pbcT2N-1M{2F*FGkiy|uCB!#Zqfd0?My9NzRvjJ})9noYTd>GgLvipq#2g4p>e*t5$qqnz2F*pz4F-I#unWYDpH_HanrQ|*0s'
            'BDrDXMcjfy?ORi|zAY-Nl`H-GTO|*)+14zk?Q{wbG2(W2<5ZGr-5qX!#z@{}hI`1edPpB)9JPyw$Pu;6Zu-F?=a6L;Z1T)D@jVdB'
            '=$?_XL5RR=V_}qH+APXZ5k}@lzR7S_RGUpmxnlbl)90x^02w;1IBo}LmWN_@Ok;0FIYx&>MG_VE0o!59bjQ?U()@bUB(!|c5ZPjC'
            '>{zN6`(od&c-Dz$Pnjf)cUv33YAnof;J}Hn3~ZW*%%MQ8?jK^lJaop!I_WxWWy{z;+15ba&da9TVs)idR2|BBSdk}F_}RHxLZWna'
            'J<o@Oa3Ai)jmtGXH9~U_9(E%>mq+V5f}3vOKO?@y3%s|y3SW9(QBf1V@Q*+o88L|!f+M~tGUo5<QH{jw%Hn0TmD>WOL}i(cw6OV@'
            '1A|MBEew}Cqr`QdYHpy0(NWq=Si`2(YS?0o8#c^q-p1mhLfQvI8<qNPy|>3wGpXHB4KP?h?kQ<WUupLf>&YXJoPM@?=6j1V-x(bP'
            'wdRCw&LT`$lsi?Qsq;VN_wE%+I6@;y_{h0bKPms6UC;MHFU#TM7Vx@UjP}{u(r?>{{Zf0@fSXnNe#`OdI}O{1Yu_rC73jF0WxQi*'
            '?!kAWVs--F!Qx%QM+X(O8B?(c{?z}b=OlF!?{Hx@Ia52_mhT!V6qc7qg3(V^KgkwhGNKw)9LPPoJ!$viczj~7Gl!t6byVdjCNHeY'
            'j}{lfT30`E+L)!f3j9L7di5Hm7>;q${xJ>|*;aEnKGyaS+K!O^Y3s>iwQ>>#>L1S3tW<&ehYf?z76ib5wOJV({so&1v~ks=AM%hp'
            'aiCr1YM&y}`0G9D%CIo>B7<lnI2cItapz##$hm*oO67jqg>$4r8}~um*IopF<C*VPFyX`%1~GO76}S3#9=m+%;NaaKTMAb+Iq_qU'
            'Zxi^*XdYq!wFA9nC0aTd@MYbWYZFL!v*Ege7am?cJb&B3;$>SGQd%4@!=u>-Ke}={inbh-u7`kXn8al3fC*q_cy}ZgQ7WS7X6*J8'
            'w7X1f?f0~q$Y#`@rbN{e8r=AZVvxxz(0CPeg#UWUUxwGZwd}EK)^`)S2cBycqy>gN<KvSj_?Uu?o#)3=-m@QVGFlAprOWrw>I5GU'
            '!-3b>RRS9gSn+HblkrnpP5|t^RkhVExc7ziT3ZESzB#2^=<O1d@tWxGIkB$0T5GWhlVndB)!(f?>z9^hZE7!gmGxf4G?W*-MV1iq'
            'K+V){kl`}imazU-Whs3>igx3W#jvSc{1wi#5nVq##Nf;8BpOL5@2Eb}xkjJ~r7-0)2Pv2G<ocJijBtLZi#2G*nDP^x>RL9sP$-b^'
            'b(PD7^+Wq|cNL2Q#kiN7+KZtszM=27rs^9*CD}ce$!BO%WDDCE!Ii*<Qlh`ER{4M?rNd`gN#a1+l*O0HWVXwyvZ5+BY?{R+V#+!l'
            '$J<>xr-1*$y%Yh}o_8PK%zk<K{Os-Q>6@po{(k=Q{6{r8_Vy+!)*{f_!1#XmyC@%Vu~S?!GbTXJ9Ajw*jby5!TUaHK46_2up;>EJ'
            'ueH5?SW<P1;V~2~tkW3^j48n^9E?CuuCqKY*WPX0->551xmR>Tfhf4efnuJOq)BjSzobtGk0sMyuDLUP2{<bCrOUtJQqH4wcq_5V'
            'c<Ejtg5!^x&<ZS_%hiFgj4$4UjOAssUR^`)!%M2>3o002PQeKnI9}j>Dc_2qV{N$RZ=dV_VJ!$uP#^ym^+kWr0;85-*(i%9!VYea'
            'y5!Zqij;N(s&3Vu1yAV#;yV2smm;{qgetj>`wLRuzoHJ7WAZty5LT<C<W|jVEeGXJoI1lPpCi59fTpF^;U@~WZuweHD5@23K)O0d'
            '_gR?xXG(1yUSi8MX)m>PKxr+{^1i3UmhHv<Mm<d>L(59riuvC2Rak}z=o(nHC0>kZTRw&@Hj+dxsb@?3bRBB`A?b~LtENuM1U5ok'
            'afDJUtHyFrn9Qxo@exMB#Mh|83M^gVnV2Iq0dW_24QDIxsHMy_)wNMC>7p6F9mZ657EnfYAGD>pP36p{`+Vw}mx{l$taX}6Qjj09'
            'v?$_KM&$H>=?$j6F-C5Yxx&#DCfRirp|DQXo-x3Fysw}k*(#h$J%?xMZFBTa9ew!w(|0dLVZWg@#jJ&8mM$wbxXIEAq&`4ev5iYb'
            '&Jrw10|&xi>#`-qO_nEIz4*x6)JH|QjL_JF6}<p6|G+)?EpYdBib=Bt#zMyFj%u*SJD3w!G+l%@KT0(YXniBdZ9-=({?8*UVg=~i'
            'A^y}RDpSr<*dd_tQR{B!qnDV29!y-~DA?Ry9nKLus-@XnZ*RG@gW&%bWz?MNIZpj3=Le@J(h?m|P*j@T%bF7MmQ}w4=cb0txlIsO'
            'fH9F$9v_JMFD?l8wT0mJ7l3P=PQmc-x^BHXF4}N7BkuuOMq81ML689SEkKbe9v@E%!}9dCkO)G6QH~mr$1@SRw8#{F#ETIMtJDBz'
            'tM*Y=!+dj04qIn*kQk{fE(>$78CB&X#YS5cKo+Pd&%WkMB#fxH7-M~hcH&T-Of=6o`U)rNQ|Y)T#2z<z{S}2&aN0hl*|CI&Vj9d^'
            'Mlt@1D(FTJ2f2wo3S{j8t?-H5dW6QdPKAv#!5cL2lkLK+bxxrTu3Xb>^|+&ck!7fFo!n1$=#{IpVzgYdqmeZxyz6Pu3U;K9_gx_6'
            'bb{Em%(czQ1y;oR3)}%npnRXkGLKhyr3NcaAy#+WjzhZTR|#|5ff7SkY0&9~#-7lrB`u>@D}c0it~CN;T!AD^d!Uf3D+&=f*v4%E'
            '@~#m##Eql@!dXT21nR5j%uO0Z*B$H4_xwK5``$vlbv3e&O1nk<(rVF3w$l38u8c$%(*98xid$N1PnF28uMXL1CgxpD|8JXOrZ(kR'
            '1)bWg`>=mzSZy8j47>p^cjiBBM~f}3>Wi=2e4Q=z+WH67V&elHr~#OAT!e6F?Lw4mKq|*nGfF55`v0nB$o>^eko^l*8+({Ue~-#z'
            '*6XhNV-G6C(>*?ze}NKYc-cvQ9hD=JH%9uX#i~U6kKQhNA_1zuia$n6xj|@T*aAWgzNoet^Wf%j_E3>CyS2shTbelzO}m-J_*<I&'
            '9wpI!=MprXg|5H49L+&RuK&eK(>U26Ez!PS(UI;fj*#A=wNqD{kV3f(NT+BFc!;s|D*1osXcVTaO~Q0%n=9BXF_0<NR*Qg56hN8F'
            '!WSgiB#)vnsR4Pk0)>GtOv|%H#AcKbaFAe96Nza_cP0sHuQ$<-<BcRP!b#(>y^U^gAvTi;Rl?%lJP90W0jJXr?c$(jP*%jLI{FMy'
            '?vLCsVbtMG7LwC{NAim~hj57~_|q=Cj2m>)8SQsub#O#R7m_F_4AlwVab<EH2cnN&idiDOa$xCD9Ui?!34=qEBj~s_tqSPmomhTs'
            'zsXWhwpo9;k@MmX`LwO<zLN>VpSRZz_Ltd0=EY|i?EPgkA%?2cI%zR(Am#r9Q4JHV'
        ),
    },
    'approval': {
        'sha256': '119794218b882b74fe3deb51c7ea08faa1cc706fe6145e48bc4a40116a202d9f',
        'encoding': 'zlib+b85',
        'data': (
            'c-p;OTXWmSl78o}n1ehBtfH9O-Ad{dT{=~YL?_%RlGl=x^{P}P2o6bvAi%}N3a!$A-><u8fB^_f%4YZY!IprTo}RvZeVKth&$|ec'
            'G>O6>R=YGWPFFh7Sx`i2qSoa$NKS*@E=#}Q^(rXzu-E(Tr_bu{r^&7Q<>vC!)mV-Hb8>rk+dJ+2(>u4bw`E?axmJO?9e+6gcz!ol'
            'd7*czSO<k#V-cO?>iqrP_#+-^RpvUw<6H${Q3i3mSGm?nk6sxL^+E+%5iK!bs6GUF9wjR^%fl?%6;okaFV60=QqNSB6e*KkmPyEa'
            'qh))RQqJ#Ws@m!xkq4H%igNH8Y<F1Fvy?p$m1)6ND~K=d@m9i2GjCC$PEWDnWtxFfvKMKpFfHQ+Hugm;45VR0bw~5CMv@j7sjR?z'
            'B=7+d!n;6a<$MmAFe6aIt!&6qmrjao4<%x=IWcEtQ4bmJ_OpSS6-7=zVXZ}Pmg!`nvuRmGu{>SqIGVFqAoL&%HDq#qQ*i~UtjaCc'
            'qUqRoqS6Ed^%tG>^cKv5;+IH2PBZ=AWt8c~O!;Z1X3HRo>6dTSEK2fnxs1XHRHlod2=KHwTYyBI?uhhs7YCtUr!hF2^`YG?-KL_p'
            'RL!ggW|i6KKf<`oqc6nqT4#FDBi@*=$F&wUk%mZo<!%>8M&q|XouB^eci*dY$-nA9NZDbU6tq2TMRuYW5cdz0>&waYkLvuxhmSYE'
            'o?ofE8+CDWeK)?oJ7W4`|6|ByUkfkOVVW)SdJ|K#+%#VYU<K+SP8QKh!(9wtmW|$gPqykCLoM_l1;iPYizrn~NEg3+i56OZf!Jo@'
            'T8EpdUEJ@_K<X4Sg`{$^R<qC&0&bZ6Nv^^&%SaA}!GX$COt;1aUki-K40@oNl}SPm@9u5~3i_r;Y+3Y>C<|vnS)^M`580)o1Z+i%'
            '$^&<_)3nTcmAqgo1-S_%kz-RyfmvaYWqTqE4Z*t}gNzh(ofd0)NcvahE>43*K2W=I9!L3F=gbss1)C#pN_6p<W*eYN_9Y5|Yo6!z'
            'SPawYvMkC>Pp2x{?$V459iGV!l=ph(8K7wmKe505o~MaSr`;40Ai=z`zeBP3`)itL8Mg}xB4CGoz^}cYv92Gk#?zl+aJTAA`5r_O'
            'YKAxRi^Ll!PXtdteM!MR3@M{uiVV<Iy%PC*{a)`IKnzLO(osZ5P~~!bF}a=GTvwVO2n+!R7g10>9l?<Tddw&Wf?|_75n+SXVU&}D'
            'Mn$gUrSiAwLX*jqS%`nPDY<pd#+{NsB&!&Y)3Vr=1*icP$-;zVQqwY8_1WS;ukuKeIWHJ&;MBBfHKU!}dv<y6VKxLgak^!s0ks0k'
            'jJn|(>uT8vn_IL4)+OLRg#mJais2dp>=-kT({KY62Sot5m={>iVaW!S7h1$kvj}pS5XrB$Y!}goJ*f9$e0vKdnSQuBzn)t0Pk$bN'
            'hO+C8kkVFgCehL|(-2pd&VhoVa*$HZgPivOkWAw!+>eMeLgj=I8S#ghH&{b1fS(p6hrlVtpFOEt!6OR{HYzqqxkE_R6i9}>>+!GS'
            'k5k7u>4(YX_#+wUlea=Z@t%p}bF0E_uowKLh;Nr?EGC5lG$$X&@Y1{h6I)VDB=%^Ak`g6z5TFt8n3qv(unHS2lZvnbMAAUuO6Syx'
            'BcNFMih5*wh=Y=uE!JtSU3yUhv&<!Hz)DWcue=209@kO0R&(rN)5;gfU^C83#1a!q5g3?k76hcrNqy(nmx@vpti?{nu^1i}Edk5-'
            'o;4ir;obw$MZ_TFJ-ov{f0?iy88c@Of&g>%K8SPR*~jsJe1ecKrys}H(9{&c>T+@iQ)5u+Dg5yY9j-a`n;L5+vtYe1ynzg9<x}=?'
            'CCV4xkh@|p1)@+swXoC!9Nm1J{5ZKjznY$Zx}4nnMFs_0rtktSX6-P9wR#J^L~;_uC46ye1WbvQUqjm>3X8rv{RjUr60)8*Hj{GB'
            'ZT6(HN(B^mjIoM=3SopHn*32CejqJ*Iw_N_E>1@3o-81L7^st4i+KDr91frHP;#Sb<J~>pJv<MY;5m}J?XK7>Ll7{P7Z!SNuCG2*'
            'R#-4_iBSfwA382*82kaT(qJ=824-o&icEi@cQltmE#fD;M~7*Pn3OuTsSMDq=6_~ho+=%H9h(ZOWgM(3AK-Kg5JC2DsJ-^b-$oL^'
            '&i)~GJ}@{}yJ)|Iu><+eh_x<XWHHwiBtQ*o;~R?c@qCgC$BW8JvC+JYIgp)l?u9Lr+rM#l$g9>YZ(vT5fjbU3grbrKQtD`gXpNNr'
            'Wuq9eQQQMjAK*Xt52ERsb8@PxURvqG_hfTkpTfe^b6+0T6YLA76D=K5GSawV-&QW9#!<CX=4r_J`NDtEkNd)@Rn}CZ?wM*bTOpy#'
            'Rjp7O-yk5f9L#j**}y-gEagxzl7F-W)0ji)Rf05>Tn3;L20LSS*3zd9*8EB=uR&im5CU&;iR>WFqE(b&i=6XM^KxE9MXYnLnx^3s'
            '>NxcCIN_8KjN=kOql4>_%Srv1WkJ&#YUjwy#3c7I7TGZShCwg|(ffgK9#Q4NI3sbe$BX7vDg;TW{ezJMc8z|`&o}CVb9%`%v#oJu'
            '!Bjz7L<7oA3W})Ax<#76%~Bj09WppIVCB74Dld?by^?n8`K|^JW;T~0A-$}gmsFH-JX25A1dm60BO{smxuYEktA72Y?~FTC=IC$>'
            '&iD=k^#t(zj#utVO}8C2ol3?%n=C;p@F|tmKK(U9RLwXqz`s>m@V<;=<s;cb#T%0U_p3I-TL0lx?g34M?odj9x_f{6cg5BF(1NxR'
            's;MJ~vbfI+6Yh9kf2flXO!{6~EKmRL_4`9CAoTryljqR(@TOFV`sw`l1HPSJ2lZ^`ziY1YFV3%Tt|u4gSL)wyQ3djn6n6X^RVjka'
            'M(=a35%L{U>AY_;h`MQ*r&25-<e`?10PkOI$<Lj!sxl1X$6%kUJ*BI8^`5C2p=El7S{`_o6PH+7QNYVc97t;*JYs<&Z%T3UG0Lf%'
            '$fazisovKMK@={-qkLn^J~HS45|#3glK5oND9|9MD2`A0fm5rwAy!ne05*|;hl_H%%WG7JGzS=xQ`aEK!zem4$q5N@N+RSwB!1Vs'
            '06`<o@-yEXP!t(?sB@vo0C~e1^d%;Fp!*k0#<2E+bx=FVr+&j<F=iqHdubZ68F1F-N%(z?kY=Vp6JDG;5_1Gq04NjzH8+tgT$jm)'
            'zzK@Z{5aUo7s1Gk98#(3fB)Tg|MoqkHHU9B!AsZ}mOIj?&y*SxGw%-tWSehXs(=fBgZ!8MexN$)16Oo5$?xbR+J1%UmT*w{V+VhO'
            '0TrofozSM4R<WkS*XGyEfZ1HD@xw6*<g1pAWJ)i$S_Zb8VrZIDx%y*hx-CU%;h6DtOQ|s)84o9wkKE1tZF2V$J&rPJa)IC6%&sf$'
            '#%!m9&&|vCldG{<mYHX&mo}G6N`6c;z?maOR}fLV;#U-HdYom98K(7H7cWTJBJR)_B?R-hR@Yt^%Gbk{cz!_QM=Eg23Ahmf0B94e'
            'G#Me1z7haba9J<85I2aZWgz)M6G0GyNw+oxw>y2cL1(lPXCMC3n@0#)TfN7Auor;E2C6Ev5)~xT{9E>{{JGx55Nm>GZQ9R$n`B!i'
            'D4A@x6>APE$ZjR!FEy|FZZ6bH$Sl2q@g&MyFv31=`trf{57Lx2=R<;I6@^r~suN4~gxq6Qs(re7haxK2V`LBp8#RZOdWS@#Jk|(T'
            '8{l&mDIhsw!d4^y8a4+XY}76&X)31~{lB^;!$z_Q)Co~MLF5PxKeQVJTKMXQx_@v9eu<E$S5dr(vZD+eH0;h9>^3?q#q@6KO_`<;'
            'qeZ(=bKZFevT|PjX%R_>2Z^2#X=vSL2nt8`f@~nv#)aXw8>qAYu1oK2(*R&)(dJDC(3B_VjSeFnf^_=xZZUFKo$)mA$n<wC)H>Ka'
            '5X7ZcDgm%V%E0f)X-;`nDkfmfr#{Q2=EEI2lgK^O4>h=@0*7F=dVLO+nMx=R9*628%`$1cNhb+TPwjCU&dYHV-z69a1-JO^H2Y)6'
            'aZocA<BWoL&^9@KD&_k9Lp3%m_2E!@crel!rxB0AXd3yaZPYX)Ydtrtbc4!ctG@+hL8W<VqBWAxO`4SXHt4cYN_?Y2>4mY!!2z5F'
            'aj_4+MheAP@;2<+fdDdQ>P_4~c~r_hih#s#>4>R*1{lxep{^eyLVHf3)nu6uWRa9&_&iNx-_9|Qr<9^Q?Tdc@nRCEL?qHG{9=ZsV'
            ')`Z!xgW0f)+3<B{yZ8;?#&1aclA%1j5oC?nJD{OH$E%G_x;&ISAGX7)SBD_$Of!*|4j29Uaeo-)QWU%vsFpxo#6=X=8I^KrzDK};'
            '-Nt1SY=cFFWTPgFG{ie52VTwLz;q`mld~IqOLvkJz-c|L*#mOOxf`z^cW%66(<c4w+;2^--fzcVokdNpA!mhiO?Adpp!Ah;qj4Nz'
            '>y7I4l5fZ)S}*vrhg8Z{buqv*vRzw>bFZCA@gXW6?WO;%!*z-@0(FFwxT<Rg_-$F#SZtM3<+I5$rPIH<)k2$r1JOTD1IA9A(h)^T'
            'j@rr><Z-{*MRKl0V8%5vo<=vw#!9@CO1pl^MG?7CdltW-1gp_Qx__`uAgOltPU90hyl*o@&GpWzvDRR}MZsa0W>?1@Ko`1O<5VmG'
            'Uu9`?V0T25VJbd`9Jm>cd<j(KY_uVG&JXy40NKy+Bi_-=+hO^?z;QE50#Jfg9PAMh)yY$Xp%cr{N&opo?a~-J+61T-Gvu`pO&nBl'
            'ZyS7V1xn99a($@pguwoRfXo$%b}}XvCY_yeXNV4oQ1E;nUoz^O`rU6@1|7lebEM+MLCRygK3aJb^~mR;bnnYECpvsuKd0<qM-Ud6'
            '$M&pJR&(6r8{iS&PGbKhl&jCMsG_$0Y0f1d*K{way1h`HxJS7|uRi&%Gc7tIh4x-NIc-E9dUJ0Y!w;?Q^XqogvYnQj(s*w6HWuk0'
            'o84)7Y$fmM>_y#NpMO^4>mSEgR}*R{vPmXTH(zFh);DQh#xV-8QAa+u^qx)z>SXwP8YRMzIhB3rKM&PM2^~ZRMw@VkMrj@@xgesd'
            'ecPSlZ4wk6$i4R^O26nd*dY6_*iZ#behVHKbp8WUaB1k3+5+9xUIP_4`WOvQt?pXK-vi}@B75JgbvI4k9!a9A`)u>Qp#sZ922DTH'
            '#TM@{qRFj3O_$*eFEa{45)jT#EX5PPM+uYXgYHNgQ?nyH&Xl=u@XBpA+Y)RZ*av{Y-<V0|(|o=MVh_3`8ZP?7j#d-3x0AgQ{j$u1'
            'SU|uWYSj1t+?zQH3A$Wm;@bo&FGK19EX$ZGe8U>S5r^#R+9lZ8!V;*9yq8wCZ1xd=r{((R$i4B%FhLhinO>ZxbSiCf2m@u3YbjqY'
            '59~6&L0aEK?2}s3dBK--ilI6u9mPTqoUf1o=XBC-u4iTXcP`Y;4&Gx38Q0l~V8{tm?!E%Vyao&Ld%hy*TwY*mey(3S;DmPGes8Gf'
            'G(=egN3ISzZA{)C6ik<?$|8$a>q4#q1d?4fFBbi&D@Fgzc40Gbc769aUC$Y&zNT53#GoiZx1qKJp%MX<+64Or1@3T*f_K<l>ln85'
            '-d1IMN9J3GhUz8>$#Fx??YwmvA~Hs(is21#w+PB^rw*$$FBnggs*%6YE4sN#segvBpio^43Js(`giT^Vs5fBYkX*JXS7lNrEQqMt'
            'ku#OgH|oON5BXvI{sxXvg=jv(mo$THNE#^Os|+lBkej!X`fubU);*|al&UVo#u@;SJFsx#oH>**%965@;p+*~|7BkAhK34<YIyqP'
            '_;PY}IT_iL0Wmx3t8jw2iK8l7R^faRk<PjTxEt{8G&n~im?M^V1ZorBC9rc(Hma|YBFeMcyQBHh-eX66w~4Mq_^zq9(%_7~BD~Nf'
            '@j6{>^+u7M@6#;jgVDoW((kJcZ8<qe9pqAtL`tr1-can2@nsSa#l_i*>txTG)Dg!W(jocI?+@ejQD<Q7Z)Xh-?DMw*#PZN&1PoZV'
            'YR(;U_I1<frA1C6<v$%^@P70!-@_P&!+uK*OV5HUjPUl;`Ss5e3l4h(V*knAhC_uKD*2WaG*lDeK{#AQU2u}8Nd^GY2|`(7dwVda'
            'G#2Z4*9CU4zzm*XF^^3V6zpaX)ZhPquO0ui*8WPJ*b<>zHXf@JNuC*edIOQQipHLs-@+sKUP}|}?Av?W&Y=3Ej`5vi=>Bz1SJp9f'
            '>B&iPwMww0F1bn15O1HH;QQRB$8)FkUEyY1rs876t%t$k8_@JdZWhorCZN8n{g`2xq>sM+o?%!PA$8VLpaZgR-)-e^5rB#OFC|&6'
            'rp%GgV5oEJc#qqnF{MqF!<mEp=9ex<i<_>#n*Ac?pk!#23i7#Ax<x5ZP**7PMVY)cteG2{K~ZP?EUxyw;HtJkA<1VR^n=*L$+LBu'
            '>KisLm7DmFU6|`=)?$3;0J=U%>9=XgJrsV;KMSpjThs4=xJ<IpESp}UX1D}3D>!rEM&*Q1)f(@J@Y`<0)}I08>h|vZYRpP-`yPBv'
            'maO1-rBRg_brDwIHKZe}^Q+q%y1ry{ngyj?;$rou8)}vE#}IN!h_4pVRRQ<%gv+g*a>0MO+B!XzhTj*ykdT(Ewev}oeAI>Cx4DkV'
            '3`Ma<c+ILn?ice%7miz-%SLi{*ZJK5%m~w~TfE)PSBkn0MW?$+>)m?|sH$oT#??s9ol%%c-QEIgd3w@E(b4zXT}vU~>$yu+Tr_&6'
            'm1`ZG6STmud0ub~URouI(0*~-tuo}WNrC91jw~m2Ky~KFEv8n8lEgu)xyelVm=6^c3V`S<V&NV^KcHf&ck~$A6=tK4ZGHHSjqBIj'
            '(w$b3mqD_LT3OXv=fO@wFCU(O%=3#@K$ne$T{j`L8tO;}U(=MyEPfncARU?f*T^zRN?5?ZCA}gO&bJ3i`qJ}l?vR{4SuUzv4miP^'
            'KC`J_o#hSHXOpp+>=R|DA@yhU_>KD2!H6=khzd@xZ|+q68Lep{3GR)WxW}y(LyobtYHHW%24J=RioyA4z!^{1_5axg;%nNiYs6Z?'
            'f~ulXEtzr-4F_)N=(JXer_D=MB4z6BF7ffA@z{a;E2t;X-GaP>&2uKJTu(EQ5mAQEa1lq1>&130PLeu{!FH<jV)M6tYumgp1oBrK'
            'z#q-W(1r79srv2w?&7EMrCf^?R^__O&8Iu(R!dY=4N|^di8?2@of2+p7@{VHmM_J{a|rX}6n0%*5VWo$^{@RHQOw>s98nX@$><_7'
            'b!+XR&a5HNnYooo-`2}rI_GP8{ydZ%m-{VhCQ)QYA#2u*WBh?pP@NXZ64GJhj};~5__l6LQX=Di5y^*92cO~}?RUSkHM|4+QS<9_'
            '>F-uwxN@JpO6A1q3=GEf?I+*BKImyST7WON8=YZwyV2IVRnKux_mhtG9A9f?SoCU#z?XZKsyYMwCv-C=*B9ff@h@Zb^ZDn?Nuxq`'
            'dKtVy|A|hAk8sw1p^sqzMRa(yZ$Sr%PVeHNLaywjvqH9S@4be*w2M_c?66pM%)uMEHq<rde^;y?5nX37`dW4SYyz&hIQ9K8-yUkT'
            '8m<*qoB!rP(3&m^O#vfQu5HZ~`eq?pl)G5p3vScj4;8dtqzQ@*WCo!gF(q<#{=%c``qrOGQ1llb<W$(N|1W@6T}_72my@kU`k0sm'
            'V+XNlKH8GA5iX78<0&~}q0=JgKM=_!!>F#39DXO@4XQ6u&g>@;T^|lqn=#*I&`y0B)Hnz|x}o|7Nw(knA1&#y3j'
        ),
    },
}

def _load_local_pipeline_modules():
    # 1) Zaten import edilebiliyorsa (yerel .venv, repo Colab'da klonlanmis) dogrudan kullan.
    try:
        import local_pipeline
        modules = {}
        for _name in _LOCAL_PIPELINE_MODULE_NAMES:
            modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
        print('local_pipeline modulleri dogrudan import edildi:', list(modules))
        return modules
    except Exception:
        pass

    # 2) Drive'da veya proje kokunun bir ustunde daha once birakilmis bir kopya var mi?
    search_roots = []
    try:
        search_roots.append(Path(DRIVE_PROJECT_ROOT))
    except NameError:
        pass
    search_roots += [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for root in search_roots:
        candidate_dir = Path(root) / 'local_pipeline'
        if all((candidate_dir / f'{_n}.py').is_file() for _n in _LOCAL_PIPELINE_MODULE_NAMES):
            sys.path.insert(0, str(candidate_dir.parent))
            try:
                modules = {}
                for _name in _LOCAL_PIPELINE_MODULE_NAMES:
                    modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
                print('local_pipeline modulleri onceden birakilmis kopyadan yuklendi:', candidate_dir)
                return modules
            except Exception:
                pass

    # 3) PROJECT_REPO_URL verilmisse git clone dene.
    if PROJECT_REPO_URL:
        clone_dir = Path('/content/youtube_automation_repo') if os.path.isdir('/content') else Path.cwd() / '_youtube_automation_repo'
        if not (clone_dir / 'local_pipeline' / (_LOCAL_PIPELINE_MODULE_NAMES[0] + '.py')).is_file():
            subprocess.run(['git', 'clone', '--depth', '1', PROJECT_REPO_URL, str(clone_dir)], check=False)
        if all((clone_dir / 'local_pipeline' / f'{_n}.py').is_file() for _n in _LOCAL_PIPELINE_MODULE_NAMES):
            sys.path.insert(0, str(clone_dir))
            try:
                modules = {}
                for _name in _LOCAL_PIPELINE_MODULE_NAMES:
                    modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
                print('local_pipeline modulleri PROJECT_REPO_URL klonundan yuklendi:', clone_dir)
                return modules
            except Exception:
                pass

    # 4) Self-contained fallback: bu hucreye gomulu, build sirasinda
    # local_pipeline/<modul>.py'den birebir uretilmis base64 kopyalari, SHA-256
    # hash'lerini dogrulayarak diske yazip import et.
    fallback_dir = Path(tempfile.mkdtemp(prefix='youtube_automation_bootstrap_'))
    pkg_dir = fallback_dir / 'local_pipeline'
    pkg_dir.mkdir(parents=True, exist_ok=True)
    (pkg_dir / '__init__.py').write_text('', encoding='utf-8')
    for _name, _entry in _LOCAL_PIPELINE_PAYLOADS.items():
        if _entry.get('encoding') != 'zlib+b85':
            raise RuntimeError(f"Desteklenmeyen gomulu payload formati: {_entry.get('encoding')!r}")
        _source_bytes = zlib.decompress(base64.b85decode(_entry['data']))
        _actual_digest = hashlib.sha256(_source_bytes).hexdigest()
        if _actual_digest != _entry['sha256']:
            raise RuntimeError(
                f"Gomulu local_pipeline.{_name} payloadi bozuk/tutarsiz: "
                f"beklenen sha256={_entry['sha256']!r}, hesaplanan={_actual_digest!r}. "
                "Bu notebook yeniden uretilmeli (nb_builder ile)."
            )
        (pkg_dir / f'{_name}.py').write_text(_source_bytes.decode('utf-8'), encoding='utf-8')
    sys.path.insert(0, str(fallback_dir))
    modules = {}
    for _name in _LOCAL_PIPELINE_MODULE_NAMES:
        modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
    print('local_pipeline modulleri: repo/Drive/PROJECT_REPO_URL bulunamadi; gomulu (self-contained, hash dogrulanmis) kopyalar kullanildi.')
    return modules

_local_pipeline_modules = _load_local_pipeline_modules()
render_utils = _local_pipeline_modules['render_utils']
video_analysis = _local_pipeline_modules['video_analysis']
approval = _local_pipeline_modules['approval']



In [ ]:
ffmpeg_bin, ffprobe_bin = render_utils.resolve_ffmpeg_binaries()
print('FFmpeg:', ffmpeg_bin)
print('System ffprobe:', ffprobe_bin or 'bulunamadi (probe_media() ffmpeg fallback kullanacak)')


def find_first_existing(candidates):
    for candidate in candidates:
        if candidate and Path(candidate).is_file():
            return Path(candidate).resolve()
    return None


## 1. Girdi dosyalarını yükle: `hot_moments.json`, `selected_script.json`, `selected_voiceover.json`

In [ ]:
def load_json_artifact(filename, override_path=''):
    candidates = [
        override_path,
        str(RESULTS_DIR / filename),
        f'results/{filename}',
        f'../results/{filename}',
        f'/content/results/{filename}',
    ]
    path = find_first_existing(candidates)
    if path is None:
        raise FileNotFoundError(
            f'{filename} bulunamadi. Aranan yollar: ' + ', '.join(c for c in candidates if c)
        )
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f), path

hot_moments_data, hot_moments_path = load_json_artifact('hot_moments.json')
selected_script, selected_script_path = load_json_artifact('selected_script.json')
selected_voiceover, selected_voiceover_path = load_json_artifact('selected_voiceover.json')

# Kanonik sema dogrulamasi (Notebook 01/02 ve master notebook ile ayni fonksiyon).
schema_errors = video_analysis.validate_hot_moments_schema(hot_moments_data)
if schema_errors:
    raise ValueError(f'hot_moments.json semasi gecersiz ({hot_moments_path}):\n- ' + '\n- '.join(schema_errors))

print('hot_moments.json      :', hot_moments_path, '| status:', hot_moments_data.get('status'))
print('selected_script.json  :', selected_script_path, '| status:', selected_script.get('status'))
print('selected_voiceover.json:', selected_voiceover_path, '| status:', selected_voiceover.get('status'))

if selected_script.get('status') != 'approved':
    print(f"UYARI: selected_script.json status='{selected_script.get('status')}' (approved degil)")
if selected_voiceover.get('status') != 'approved':
    print(f"UYARI: selected_voiceover.json status='{selected_voiceover.get('status')}' (approved degil)")
print('Not: onaylanmamis girdilerle render yine calisir (inceleme icin) ama nihai durum')
print('     review_required/failed kalir, ASLA approved/production_ready olmaz.')


## 2. Render yapılandırması ve onay bayrağı

Tüm render mantığı tek bir kanonik çağrıda (`render_utils.run_render_pipeline`) toplanmıştır: aralık kesme, altyazı hizalama (gerçek seslendirme sesinden `faster-whisper` ile; başarısız olursa orantılı fallback), dikey kadraj + altyazı yakma + ses miksleme (tek `ffmpeg` geçişi), teknik doğrulama ve onay kapısı değerlendirmesi.

In [ ]:
RENDER_MODE = 'blurred_background'  # 'blurred_background' veya 'center_crop'
TARGET_WIDTH = 1080
TARGET_HEIGHT = 1920
PREVIEW_MODE = True  # True = hizli/dusuk kalite onizleme; False = uretim kalitesi

DUCKING_ENABLED = True
ORIGINAL_AUDIO_BASE_VOLUME = 0.35
NARRATION_VOLUME = 1.0
DUCK_RATIO = 8

SUBTITLE_FONTSIZE = 20
SUBTITLE_MARGIN_V = 90
SUBTITLE_MAX_CHARS_PER_CUE = 42

SOURCE_VIDEO_PATH_OVERRIDE = ''  # zorla belirli bir ham kaynak video vermek istersen buraya yaz
print('Render modu:', RENDER_MODE, '| onizleme mi:', PREVIEW_MODE)
print("NOT: onay bu hucrede DEGIL, bolum 5'te (videoyu izledikten sonra) verilir.")


## 3. Render çalıştır (kanonik `render_utils.run_render_pipeline`)

In [ ]:
render_manifest = render_utils.run_render_pipeline(
    hot_moments_data, selected_script, selected_voiceover,
    hot_moments_path, selected_script_path, selected_voiceover_path,
    render_dir=RENDER_DIR,
    render_mode=RENDER_MODE, target_width=TARGET_WIDTH, target_height=TARGET_HEIGHT,
    preview_mode=PREVIEW_MODE, ducking_enabled=DUCKING_ENABLED,
    original_audio_base_volume=ORIGINAL_AUDIO_BASE_VOLUME, narration_volume=NARRATION_VOLUME,
    duck_ratio=DUCK_RATIO, subtitle_fontsize=SUBTITLE_FONTSIZE,
    subtitle_margin_v=SUBTITLE_MARGIN_V, subtitle_max_chars_per_cue=SUBTITLE_MAX_CHARS_PER_CUE,
    source_video_path_override=SOURCE_VIDEO_PATH_OVERRIDE,
)

print('Kaynak cozumlemesi:', render_manifest['inputs']['source_description'])
print('Altyazi kaynagi:', render_manifest['subtitles']['source'],
      f"({render_manifest['subtitles']['cue_count']} satir)")
print('Render dosyasi:', render_manifest['output']['final_video_path'],
      f"({render_manifest['output']['render_latency_sec']}s)")
print()
print('Teknik dogrulama (passed):', render_manifest['output']['validation']['passed'],
      '| probe backend:', render_manifest['output']['validation']['details']['probe_backend'])
print(json.dumps(render_manifest['output']['validation']['checks'], indent=2))
print()
print('Onay oncesi kosullar:', json.dumps(render_manifest['approval']['conditions'], indent=2))
if render_manifest['approval']['failed_conditions']:
    print('Saglanmayan kosul(lar):', render_manifest['approval']['failed_conditions'])
print('Render durumu:', render_manifest['status'], '-', render_manifest['status_reason'])


## 5. Render'ı izle ve ONAYLA (ayrı adım)

Videoyu **izledikten sonra** aşağıda yazdırılan SHA-256'yı `RENDER_REVIEWED_SHA256` alanına yapıştır.

Render onayı yalnızca **tüm** koşullar sağlandığında verilebilir: teknik doğrulama geçmiş **ve**
script onaylı **ve** seslendirme onaylı **ve** seslendirme-script hash eşleşmesi sağlanmış olmalı.
Herhangi biri eksikse onay reddedilir ve artefakt `review_required` kalır.

Sadece izlediğini kaydetmek istersen `RECORD_RENDER_REVIEW_ONLY = True` yap — bu `status`'ü değiştirmez.

In [ ]:
video_path = render_manifest['output']['final_video_path']
try:
    from IPython.display import Video, display as _display
    _display(Video(video_path, embed=True, width=320))
except Exception as exc:
    print('Video oynatici gosterilemedi:', exc)
    print('Dosyayi elle izle:', video_path)

render_review_info = approval.describe_for_review('render', render_manifest)
print('Icerik SHA-256:', render_review_info['content_sha256'])
if not render_review_info['reviewable']:
    print()
    print('!!! FINAL ONAY ENGELLI !!!')
    print(render_review_info['blocker'])
    for condition in render_review_info.get('blocking_conditions', []):
        print('  -', condition)

RENDER_REVIEWED_SHA256 = ''   # yukaridaki SHA-256'yi buraya yapistir
RENDER_APPROVED_BY = ''
RENDER_APPROVAL_NOTES = ''
RECORD_RENDER_REVIEW_ONLY = False  # True: sadece "izledim" kaydi, status DEGISMEZ

if not RENDER_REVIEWED_SHA256 or not RENDER_APPROVED_BY:
    print('Onay ATLANDI: RENDER_REVIEWED_SHA256 ve RENDER_APPROVED_BY doldurulmali.')
    print('status olarak kaliyor:', render_manifest['status'])
elif RECORD_RENDER_REVIEW_ONLY:
    try:
        render_manifest = approval.record_render_review(
            render_manifest, RENDER_REVIEWED_SHA256, RENDER_APPROVED_BY, RENDER_APPROVAL_NOTES,
        )
        print('Inceleme kaydedildi (render_review). status DEGISMEDI:', render_manifest['status'])
    except ValueError as exc:
        print('INCELEME KAYDI REDDEDILDI:', exc)
else:
    try:
        render_manifest = approval.approve_artifact(
            render_manifest, 'render', RENDER_REVIEWED_SHA256,
            RENDER_APPROVED_BY, RENDER_APPROVAL_NOTES,
        )
        print('Render ONAYLANDI (status:', render_manifest['status'], ')')
        print('NOT: onay yalnizca bir kayittir; bu notebook hicbir sey yayinlamaz.')
    except ValueError as exc:
        print('ONAY REDDEDILDI:', exc)
        print('status olarak kaliyor:', render_manifest['status'])
        print("Sadece izledigini kaydetmek istersen RECORD_RENDER_REVIEW_ONLY = True yap.")


## 6. Dışa aktarım: `render_manifest.json`

In [ ]:
render_manifest_path = RENDER_DIR / 'render_manifest.json'
with open(render_manifest_path, 'w', encoding='utf-8') as f:
    json.dump(render_manifest, f, ensure_ascii=False, indent=2)
print('Kaydedildi:', render_manifest_path, '(status:', render_manifest['status'], ')')

render_validation_path = RENDER_DIR / 'render_validation.json'
with open(render_validation_path, 'w', encoding='utf-8') as f:
    json.dump(render_manifest['output']['validation'], f, ensure_ascii=False, indent=2)
print('Kaydedildi:', render_validation_path)

print('HATIRLATMA: bu notebook hicbir yere yukleme yapmadi (not_uploaded='
      f"{render_manifest['not_uploaded']}).")
print('Yayin, ayri ve elle onayli bir adimdir.')
